# Systematic Evaluation and Enhancement of bio.tools Annotation Quality

## Overview

The quality of metadata in the ELIXIR bio.tools registry affects its usability for both end users and developers. However, the completeness and consistency of tool annotations vary significantly. This notebook investigates the current state of metadata quality and proposes a systematic framework to assess and improve it.

### Key Objectives
1. **Implement a scoring pipeline** to classify tools into annotation quality tiers (1-5)
2. **Analyze completeness patterns** across tool collections and domains
3. **Identify frequently missing attributes** and structural issues
4. **Integrate bio.tools linter output** into quality assessment
5. **Propose revisions** to the Tool Information Standards

### Expected Outcomes
- A comprehensive scoring and analysis pipeline for bio.tools metadata evaluation
- Visual and tabular reports highlighting completeness metrics per tier and domain
- An integrated dataset combining scoring, linter results, and tool metadata
- Evidence-based recommendations for improving Tool Information Standards

---

## 1. Import Required Libraries and Define Constants

First, we'll import all necessary Python libraries for data handling, API requests, JSON schema validation, and visualization.

In [ ]:
# Import required libraries
import sys
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# Add project source to path
project_root = Path('.').parent
sys.path.insert(0, str(project_root / 'src'))

# Import project modules
try:
    from data_collection.api_client import BioToolsAPIClient
    from data_collection.data_parser import BioToolsDataParser
    from scoring.completeness_scorer import CompletenessScorer
    from scoring.tier_classifier import TierClassifier
    from analysis.statistics import QualityStatistics
    from visualization.charts import QualityVisualizer
    print("✓ Successfully imported project modules")
except ImportError as e:
    print(f"⚠ Warning: Could not import project modules: {e}")
    print("Will use simplified implementations for demonstration")

# Define constants
API_BASE_URL = "https://bio.tools/api/tool/"
SCHEMA_URL = "https://github.com/bio-tools/biotoolsSchema/raw/master/versions/biotools-3.0.0.json"
CONFIG_PATH = "../config/scoring_config.yaml"
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../data/processed")
VIZ_DIR = Path("../data/visualizations")

# Create directories if they don't exist
for dir_path in [DATA_DIR, RESULTS_DIR, VIZ_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✓ Libraries imported and constants defined")
print(f"✓ Data directories created: {DATA_DIR}, {RESULTS_DIR}, {VIZ_DIR}")

## 2. Retrieve Tool Metadata from bio.tools API

We'll fetch tool entries from the bio.tools API. For this demonstration, we'll focus on proteomics tools as they represent a well-established domain with varying annotation quality.

In [ ]:
# Initialize API client
try:
    api_client = BioToolsAPIClient()
    print("✓ API client initialized successfully")
except:
    print("⚠ Using simplified API client implementation")
    
    class SimpleAPIClient:
        def __init__(self):
            self.session = requests.Session()
            self.session.headers.update({
                'User-Agent': 'bio.tools-quality-evaluation/1.0',
                'Content-Type': 'application/json'
            })
        
        def get_tools_by_topic(self, topic, limit=50):
            url = f"{API_BASE_URL}"
            params = {'topic': topic, 'page_size': min(limit, 100)}
            tools = []
            page = 1
            
            while len(tools) < limit:
                params['page'] = page
                try:
                    response = self.session.get(url, params=params, timeout=30)
                    response.raise_for_status()
                    data = response.json()
                    
                    page_tools = data.get('list', [])
                    if not page_tools:
                        break
                    
                    tools.extend(page_tools)
                    if len(page_tools) < params['page_size']:
                        break
                    page += 1
                    
                except Exception as e:
                    print(f"Error fetching data: {e}")
                    break
            
            return tools[:limit]
    
    api_client = SimpleAPIClient()

# Collect proteomics tools for analysis
print("Fetching proteomics tools from bio.tools API...")
proteomics_tools = api_client.get_tools_by_topic("Proteomics", limit=100)

print(f"✓ Retrieved {len(proteomics_tools)} proteomics tools")

# Display basic information about the first few tools
if proteomics_tools:
    print("\nSample tools retrieved:")
    for i, tool in enumerate(proteomics_tools[:5]):
        name = tool.get('name', 'Unknown')
        biotools_id = tool.get('biotoolsID', 'Unknown')
        description = tool.get('description', 'No description')[:100]
        print(f"{i+1}. {name} ({biotools_id})")
        print(f"   Description: {description}...")
        print()
else:
    print("⚠ No tools retrieved. Using sample data for demonstration.")

## 3. Validate Metadata Against biotoolsSchema

We'll use JSON schema validation to check each tool's metadata for compliance with the biotoolsSchema, identifying structural issues.

In [ ]:
# Simple schema validation implementation
def validate_tool_schema(tool_data):
    """Simple validation check for essential fields."""
    validation_result = {
        'is_valid': True,
        'errors': [],
        'warnings': []
    }
    
    # Check required fields
    required_fields = ['biotoolsID', 'name', 'description', 'homepage']
    for field in required_fields:
        if field not in tool_data or not tool_data[field]:
            validation_result['errors'].append(f"Missing required field: {field}")
            validation_result['is_valid'] = False
    
    # Check optional but important fields
    important_fields = ['function', 'topic', 'toolType', 'license']
    for field in important_fields:
        if field not in tool_data or not tool_data[field]:
            validation_result['warnings'].append(f"Missing important field: {field}")
    
    return validation_result

# Validate all tools
print("Validating tool metadata against schema requirements...")
validation_results = []

for tool in proteomics_tools:
    biotools_id = tool.get('biotoolsID', 'unknown')
    validation = validate_tool_schema(tool)
    validation['biotoolsID'] = biotools_id
    validation_results.append(validation)

# Summary of validation results
valid_tools = sum(1 for r in validation_results if r['is_valid'])
invalid_tools = len(validation_results) - valid_tools

print(f"✓ Validation complete:")
print(f"  - Valid tools: {valid_tools}")
print(f"  - Invalid tools: {invalid_tools}")
print(f"  - Validation rate: {valid_tools/len(validation_results)*100:.1f}%")

# Most common validation errors
all_errors = []
all_warnings = []
for result in validation_results:
    all_errors.extend(result['errors'])
    all_warnings.extend(result['warnings'])

print(f"\nMost common errors:")
error_counts = Counter(all_errors)
for error, count in error_counts.most_common(5):
    print(f"  - {error}: {count} tools")

print(f"\nMost common warnings:")
warning_counts = Counter(all_warnings)
for warning, count in warning_counts.most_common(5):
    print(f"  - {warning}: {count} tools")

## 4. Implement Tier-Based Scoring System

We'll implement a comprehensive scoring system that maps Tool Information Standards to a 5-tier classification system based on metadata richness and completeness.

In [ ]:
# Define scoring configuration
SCORING_CONFIG = {
    'weights': {
        'basic_info': 20,        # Name, description, homepage, biotoolsID
        'detailed_description': 15,  # Function, topic, operation
        'technical_details': 25,    # Version, language, OS, license
        'documentation': 20,       # Documentation links, publications
        'accessibility': 10,       # Download links, repositories
        'community': 10          # Credits, contact information
    },
    'tiers': {
        1: (0, 20),      # Minimal annotation
        2: (21, 40),     # Basic annotation
        3: (41, 60),     # Moderate annotation
        4: (61, 80),     # Good annotation
        5: (81, 100)     # Excellent annotation
    }
}

def score_basic_info(tool_data):
    """Score basic information completeness (max 20 points)."""
    fields = ['name', 'description', 'homepage', 'biotoolsID', 'version']
    present_count = sum(1 for field in fields if field in tool_data and tool_data[field])
    return (present_count / len(fields)) * SCORING_CONFIG['weights']['basic_info']

def score_detailed_description(tool_data):
    """Score detailed description completeness (max 15 points)."""
    score = 0
    max_score = SCORING_CONFIG['weights']['detailed_description']
    
    # Function information
    if 'function' in tool_data and tool_data['function']:
        score += max_score * 0.4
    
    # Topic information
    if 'topic' in tool_data and tool_data['topic']:
        score += max_score * 0.3
    
    # Operation information (from functions)
    if 'function' in tool_data:
        for func in tool_data['function']:
            if 'operation' in func and func['operation']:
                score += max_score * 0.3
                break
    
    return min(score, max_score)

def score_technical_details(tool_data):
    """Score technical details completeness (max 25 points)."""
    fields = ['toolType', 'language', 'operatingSystem', 'license', 'maturity']
    present_count = sum(1 for field in fields if field in tool_data and tool_data[field])
    return (present_count / len(fields)) * SCORING_CONFIG['weights']['technical_details']

def score_documentation(tool_data):
    """Score documentation completeness (max 20 points)."""
    score = 0
    max_score = SCORING_CONFIG['weights']['documentation']
    
    # Documentation links
    if 'documentation' in tool_data and tool_data['documentation']:
        score += max_score * 0.5
    
    # Publications
    if 'publication' in tool_data and tool_data['publication']:
        score += max_score * 0.5
    
    return score

def score_accessibility(tool_data):
    """Score accessibility completeness (max 10 points)."""
    score = 0
    max_score = SCORING_CONFIG['weights']['accessibility']
    
    # Download links
    if 'download' in tool_data and tool_data['download']:
        score += max_score * 0.6
    
    # Repository links
    if 'repository' in tool_data and tool_data['repository']:
        score += max_score * 0.4
    
    return score

def score_community(tool_data):
    """Score community information completeness (max 10 points)."""
    score = 0
    max_score = SCORING_CONFIG['weights']['community']
    
    # Contact information
    if 'contact' in tool_data and tool_data['contact']:
        score += max_score * 0.6
    
    # Credit information
    if 'credit' in tool_data and tool_data['credit']:
        score += max_score * 0.4
    
    return score

def calculate_total_score(tool_data):
    """Calculate total score and determine tier."""
    scores = {
        'basic_info': score_basic_info(tool_data),
        'detailed_description': score_detailed_description(tool_data),
        'technical_details': score_technical_details(tool_data),
        'documentation': score_documentation(tool_data),
        'accessibility': score_accessibility(tool_data),
        'community': score_community(tool_data)
    }
    
    total_score = sum(scores.values())
    
    # Determine tier
    tier = 1
    for t, (min_score, max_score) in SCORING_CONFIG['tiers'].items():
        if min_score <= total_score <= max_score:
            tier = t
            break
    
    return {
        'total_score': round(total_score, 2),
        'tier': tier,
        'component_scores': {k: round(v, 2) for k, v in scores.items()},
        'biotoolsID': tool_data.get('biotoolsID', 'unknown'),
        'name': tool_data.get('name', 'unknown')
    }

# Score all tools
print("Scoring tools using tier-based system...")
scoring_results = []

for tool in proteomics_tools:
    score_result = calculate_total_score(tool)
    scoring_results.append(score_result)

print(f"✓ Scored {len(scoring_results)} tools")

# Display scoring summary
tier_counts = Counter(result['tier'] for result in scoring_results)
total_scores = [result['total_score'] for result in scoring_results]

print(f"\nScoring Summary:")
print(f"  - Average score: {np.mean(total_scores):.2f}/100")
print(f"  - Score range: {min(total_scores):.2f} - {max(total_scores):.2f}")
print(f"\nTier Distribution:")
for tier in range(1, 6):
    count = tier_counts.get(tier, 0)
    percentage = (count / len(scoring_results)) * 100
    tier_names = {1: 'Minimal', 2: 'Basic', 3: 'Moderate', 4: 'Good', 5: 'Excellent'}
    print(f"  - Tier {tier} ({tier_names[tier]}): {count} tools ({percentage:.1f}%)")

# Show examples from each tier
print(f"\nExample tools from each tier:")
for tier in range(1, 6):
    tier_tools = [r for r in scoring_results if r['tier'] == tier]
    if tier_tools:
        example = tier_tools[0]
        print(f"  Tier {tier}: {example['name']} (Score: {example['total_score']})")

## 5. Analyze Completeness Patterns Across Tool Collections

Let's aggregate and visualize completeness scores by different dimensions to identify trends and gaps in annotation quality.

In [ ]:
# Create comprehensive analysis dataframe
df_analysis = pd.DataFrame(scoring_results)

# Add tool metadata for analysis
for i, tool in enumerate(proteomics_tools):
    if i < len(df_analysis):
        df_analysis.loc[i, 'tool_type'] = ', '.join(tool.get('toolType', ['Unknown']))
        df_analysis.loc[i, 'has_license'] = bool(tool.get('license'))
        df_analysis.loc[i, 'has_publication'] = bool(tool.get('publication'))
        df_analysis.loc[i, 'has_documentation'] = bool(tool.get('documentation'))
        df_analysis.loc[i, 'language_count'] = len(tool.get('language', []))
        df_analysis.loc[i, 'function_count'] = len(tool.get('function', []))

# Component score analysis
component_stats = df_analysis[['basic_info', 'detailed_description', 'technical_details', 
                              'documentation', 'accessibility', 'community']].describe()

print("Component Score Statistics:")
print(component_stats.round(2))

# Identify the weakest components
component_means = {
    'Basic Info': df_analysis['basic_info'].mean(),
    'Description': df_analysis['detailed_description'].mean(),
    'Technical': df_analysis['technical_details'].mean(),
    'Documentation': df_analysis['documentation'].mean(),
    'Accessibility': df_analysis['accessibility'].mean(),
    'Community': df_analysis['community'].mean()
}

print(f"\nComponent Ranking (by average score):")
sorted_components = sorted(component_means.items(), key=lambda x: x[1], reverse=True)
for i, (component, score) in enumerate(sorted_components, 1):
    max_scores = {'Basic Info': 20, 'Description': 15, 'Technical': 25, 
                  'Documentation': 20, 'Accessibility': 10, 'Community': 10}
    max_score = max_scores.get(component, 20)
    percentage = (score / max_score) * 100
    print(f"  {i}. {component}: {score:.2f}/{max_score} ({percentage:.1f}%)")

# Analyze by tool type
tool_type_analysis = df_analysis.groupby('tool_type').agg({
    'total_score': ['mean', 'count'],
    'tier': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]
}).round(2)

print(f"\nAnalysis by Tool Type:")
print(tool_type_analysis)

# Field completeness analysis
field_completeness = {}
important_fields = ['license', 'publication', 'documentation', 'language', 'function']

for field in important_fields:
    if field == 'language':
        completeness = (df_analysis['language_count'] > 0).sum()
    elif field == 'function':
        completeness = (df_analysis['function_count'] > 0).sum()
    else:
        completeness = df_analysis[f'has_{field}'].sum()
    
    field_completeness[field] = {
        'count': completeness,
        'percentage': (completeness / len(df_analysis)) * 100
    }

print(f"\nField Completeness Analysis:")
for field, stats in sorted(field_completeness.items(), key=lambda x: x[1]['percentage'], reverse=True):
    print(f"  - {field.title()}: {stats['count']}/{len(df_analysis)} ({stats['percentage']:.1f}%)")

# Identify improvement opportunities
low_tier_tools = df_analysis[df_analysis['tier'] <= 2]
improvement_opportunities = len(low_tier_tools)
close_to_upgrade = len(df_analysis[(df_analysis['tier'] < 5) & 
                                  (df_analysis['total_score'] > df_analysis['total_score'].quantile(0.75))])

print(f"\nImprovement Opportunities:")
print(f"  - Tools in Tier 1-2 (need significant improvement): {improvement_opportunities}")
print(f"  - Tools close to tier upgrade: {close_to_upgrade}")
print(f"  - Overall improvement potential: {(improvement_opportunities/len(df_analysis)*100):.1f}% of tools")

## 6. Integrate and Parse bio.tools Linter Output

We'll simulate linter output analysis to identify structural and syntactic issues in tool annotations. In a real implementation, this would integrate with the actual bio.tools linter.

In [ ]:
# Simulate linter analysis (in real implementation, this would call the actual bio.tools linter)
def simulate_linter_analysis(tool_data):
    """Simulate linter output for demonstration purposes."""
    errors = []
    warnings = []
    
    biotools_id = tool_data.get('biotoolsID', 'unknown')
    
    # Check for common structural issues
    if not tool_data.get('name'):
        errors.append("missing_required_field:name")
    
    if not tool_data.get('description'):
        errors.append("missing_required_field:description")
    
    if not tool_data.get('homepage'):
        warnings.append("missing_important_field:homepage")
    
    # Check URL validity (simplified)
    homepage = tool_data.get('homepage', '')
    if homepage and not (homepage.startswith('http://') or homepage.startswith('https://')):
        warnings.append("invalid_url:homepage")
    
    # Check license format
    license_info = tool_data.get('license')
    if license_info and not isinstance(license_info, str):
        warnings.append("invalid_license_format")
    
    # Check function structure
    functions = tool_data.get('function', [])
    if not functions:
        warnings.append("missing_function_information")
    else:
        for i, func in enumerate(functions):
            if not func.get('operation'):
                warnings.append(f"missing_operation_in_function_{i}")
    
    # Check topic information
    topics = tool_data.get('topic', [])
    if not topics:
        warnings.append("missing_topic_information")
    
    # Check publication format
    publications = tool_data.get('publication', [])
    for pub in publications:
        if isinstance(pub, dict):
            if not pub.get('doi') and not pub.get('pmid'):
                warnings.append("incomplete_publication_reference")
    
    return {
        'biotoolsID': biotools_id,
        'error_count': len(errors),
        'warning_count': len(warnings),
        'errors': errors,
        'warnings': warnings,
        'linter_score': max(0, 100 - (len(errors) * 10) - (len(warnings) * 2))
    }

# Run linter analysis on all tools
print("Running linter analysis on all tools...")
linter_results = []

for tool in proteomics_tools:
    linter_result = simulate_linter_analysis(tool)
    linter_results.append(linter_result)

print(f"✓ Linter analysis complete for {len(linter_results)} tools")

# Analyze linter results
total_errors = sum(result['error_count'] for result in linter_results)
total_warnings = sum(result['warning_count'] for result in linter_results)
avg_linter_score = np.mean([result['linter_score'] for result in linter_results])

print(f"\nLinter Analysis Summary:")
print(f"  - Total errors: {total_errors}")
print(f"  - Total warnings: {total_warnings}")
print(f"  - Average linter score: {avg_linter_score:.2f}/100")
print(f"  - Tools with errors: {sum(1 for r in linter_results if r['error_count'] > 0)}")
print(f"  - Tools with warnings: {sum(1 for r in linter_results if r['warning_count'] > 0)}")

# Most common linter issues
all_errors = []
all_warnings = []
for result in linter_results:
    all_errors.extend(result['errors'])
    all_warnings.extend(result['warnings'])

print(f"\nMost Common Errors:")
error_counts = Counter(all_errors)
for error, count in error_counts.most_common(5):
    print(f"  - {error}: {count} tools")

print(f"\nMost Common Warnings:")
warning_counts = Counter(all_warnings)
for warning, count in warning_counts.most_common(5):
    print(f"  - {warning}: {count} tools")

# Tools with highest error rates
problematic_tools = sorted(linter_results, key=lambda x: x['error_count'] + x['warning_count'], reverse=True)
print(f"\nTools with most linter issues:")
for i, tool in enumerate(problematic_tools[:5]):
    print(f"  {i+1}. {tool['biotoolsID']}: {tool['error_count']} errors, {tool['warning_count']} warnings")

## 7. Merge Completeness Scores with Linter Diagnostics

Now we'll combine our scoring results with linter diagnostics to create an integrated quality assessment for each tool.

In [ ]:
# Create integrated quality assessment
integrated_results = []

for score_result, linter_result in zip(scoring_results, linter_results):
    # Ensure we're matching the same tools
    if score_result['biotoolsID'] == linter_result['biotoolsID']:
        
        # Calculate composite quality score
        completeness_score = score_result['total_score']
        linter_score = linter_result['linter_score']
        
        # Weighted composite score (70% completeness, 30% linter)
        composite_score = (completeness_score * 0.7) + (linter_score * 0.3)
        
        # Determine overall quality grade
        if composite_score >= 80:
            quality_grade = 'A'
        elif composite_score >= 65:
            quality_grade = 'B'
        elif composite_score >= 50:
            quality_grade = 'C'
        elif composite_score >= 35:
            quality_grade = 'D'
        else:
            quality_grade = 'F'
        
        # Create integrated result
        integrated_result = {
            'biotoolsID': score_result['biotoolsID'],
            'name': score_result['name'],
            'tier': score_result['tier'],
            'completeness_score': completeness_score,
            'linter_score': linter_score,
            'composite_score': round(composite_score, 2),
            'quality_grade': quality_grade,
            'error_count': linter_result['error_count'],
            'warning_count': linter_result['warning_count'],
            'component_scores': score_result['component_scores'],
            'top_issues': linter_result['errors'][:3] + linter_result['warnings'][:3]
        }
        
        integrated_results.append(integrated_result)

print(f"✓ Created integrated quality assessment for {len(integrated_results)} tools")

# Create comprehensive DataFrame
df_integrated = pd.DataFrame(integrated_results)

# Quality distribution analysis
grade_distribution = df_integrated['quality_grade'].value_counts().sort_index()
print(f"\nQuality Grade Distribution:")
for grade, count in grade_distribution.items():
    percentage = (count / len(df_integrated)) * 100
    print(f"  Grade {grade}: {count} tools ({percentage:.1f}%)")

# Correlation analysis
correlation_matrix = df_integrated[['completeness_score', 'linter_score', 'composite_score', 
                                   'error_count', 'warning_count']].corr()

print(f"\nCorrelation Analysis:")
print(f"  Completeness vs Linter Score: {correlation_matrix.loc['completeness_score', 'linter_score']:.3f}")
print(f"  Completeness vs Error Count: {correlation_matrix.loc['completeness_score', 'error_count']:.3f}")
print(f"  Linter Score vs Warning Count: {correlation_matrix.loc['linter_score', 'warning_count']:.3f}")

# Identify tools with discrepancies (high completeness but many linter issues)
discrepancy_tools = df_integrated[
    (df_integrated['completeness_score'] > 60) & 
    (df_integrated['error_count'] + df_integrated['warning_count'] > 5)
]

print(f"\nTools with Score Discrepancies:")
print(f"  (High completeness but many linter issues): {len(discrepancy_tools)} tools")
if len(discrepancy_tools) > 0:
    for _, tool in discrepancy_tools.head(3).iterrows():
        print(f"    - {tool['name']}: Completeness {tool['completeness_score']:.1f}, "
              f"{tool['error_count']} errors, {tool['warning_count']} warnings")

# Best and worst performing tools
best_tools = df_integrated.nlargest(5, 'composite_score')
worst_tools = df_integrated.nsmallest(5, 'composite_score')

print(f"\nTop 5 Highest Quality Tools:")
for _, tool in best_tools.iterrows():
    print(f"  - {tool['name']} (Grade {tool['quality_grade']}, Score: {tool['composite_score']})")

print(f"\nTop 5 Tools Needing Improvement:")
for _, tool in worst_tools.iterrows():
    print(f"  - {tool['name']} (Grade {tool['quality_grade']}, Score: {tool['composite_score']})")
    if tool['top_issues']:
        print(f"    Issues: {', '.join(tool['top_issues'][:2])}")

# Summary statistics
print(f"\nIntegrated Quality Assessment Summary:")
print(f"  - Average composite score: {df_integrated['composite_score'].mean():.2f}/100")
print(f"  - Score standard deviation: {df_integrated['composite_score'].std():.2f}")
print(f"  - Tools with Grade A or B: {len(df_integrated[df_integrated['quality_grade'].isin(['A', 'B'])])}/{len(df_integrated)}")
print(f"  - Tools needing improvement (Grade D or F): {len(df_integrated[df_integrated['quality_grade'].isin(['D', 'F'])])}/{len(df_integrated)}")

## 8. Generate Visual Summaries (Radar Charts, Heatmaps)

We'll create comprehensive visualizations including radar charts and heatmaps to summarize our findings across tools and quality dimensions.

In [ ]:
# Set up visualization style
plt.style.use('default')
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

# 1. Tier Distribution Pie Chart
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

tier_counts = df_integrated['tier'].value_counts().sort_index()
tier_labels = [f'Tier {t}\n({tier_counts[t]} tools)' for t in tier_counts.index]
ax1.pie(tier_counts.values, labels=tier_labels, autopct='%1.1f%%', colors=colors[:len(tier_counts)])
ax1.set_title('Distribution of Tools by Quality Tier', fontsize=14, fontweight='bold')

# 2. Quality Grade Distribution
grade_counts = df_integrated['quality_grade'].value_counts().sort_index()
bars = ax2.bar(grade_counts.index, grade_counts.values, color=colors[:len(grade_counts)])
ax2.set_title('Distribution by Quality Grade', fontsize=14, fontweight='bold')
ax2.set_xlabel('Quality Grade')
ax2.set_ylabel('Number of Tools')
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom')

# 3. Component Score Comparison
component_data = []
component_names = ['Basic Info', 'Description', 'Technical', 'Documentation', 'Accessibility', 'Community']
component_fields = ['basic_info', 'detailed_description', 'technical_details', 
                   'documentation', 'accessibility', 'community']
max_scores = [20, 15, 25, 20, 10, 10]

for i, field in enumerate(component_fields):
    scores = [result['component_scores'][field] for result in integrated_results]
    component_data.append(scores)

box_plot = ax3.boxplot(component_data, labels=component_names, patch_artist=True)
for patch, color in zip(box_plot['boxes'], colors):
    patch.set_facecolor(color)
ax3.set_title('Component Score Distribution', fontsize=14, fontweight='bold')
ax3.set_ylabel('Score')
ax3.tick_params(axis='x', rotation=45)

# 4. Score vs Error Count Scatter
scatter = ax4.scatter(df_integrated['error_count'], df_integrated['composite_score'], 
                     c=df_integrated['tier'], cmap='viridis', alpha=0.7)
ax4.set_xlabel('Error Count')
ax4.set_ylabel('Composite Score')
ax4.set_title('Composite Score vs Error Count', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=ax4, label='Tier')

plt.tight_layout()
plt.show()

# Create radar chart for average component scores by tier
def create_radar_chart():
    # Calculate average component scores by tier
    tier_averages = df_integrated.groupby('tier')[
        [f'component_scores.{field}' for field in component_fields]
    ].mean()
    
    # For each tier, we need to extract component scores differently
    tier_component_averages = {}
    for tier in range(1, 6):
        tier_tools = [r for r in integrated_results if r['tier'] == tier]
        if tier_tools:
            tier_avg = {}
            for field in component_fields:
                scores = [tool['component_scores'][field] for tool in tier_tools]
                tier_avg[field] = np.mean(scores)
            tier_component_averages[tier] = tier_avg
    
    # Create radar chart
    angles = np.linspace(0, 2 * np.pi, len(component_names), endpoint=False).tolist()
    angles += angles[:1]  # Close the circle
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    
    for tier, avg_scores in tier_component_averages.items():
        values = [avg_scores[field] for field in component_fields]
        values += values[:1]  # Close the circle
        
        ax.plot(angles, values, 'o-', linewidth=2, label=f'Tier {tier}', color=colors[tier-1])
        ax.fill(angles, values, alpha=0.25, color=colors[tier-1])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(component_names)
    ax.set_ylim(0, max(max_scores))
    ax.set_title('Average Component Scores by Tier', size=16, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

create_radar_chart()

# Create heatmap of individual tool scores
def create_quality_heatmap():
    # Prepare data for heatmap
    heatmap_data = []
    tool_names = []
    
    # Select top 20 tools for visualization
    top_tools = df_integrated.nlargest(20, 'composite_score')
    
    for _, tool in top_tools.iterrows():
        tool_scores = []
        # Get component scores
        for field in component_fields:
            # Find the original result for this tool
            original_result = next(r for r in integrated_results if r['biotoolsID'] == tool['biotoolsID'])
            tool_scores.append(original_result['component_scores'][field])
        
        heatmap_data.append(tool_scores)
        tool_names.append(tool['name'][:20] + '...' if len(tool['name']) > 20 else tool['name'])
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, 10))
    
    heatmap = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto')
    
    # Set ticks and labels
    ax.set_xticks(range(len(component_names)))
    ax.set_xticklabels(component_names, rotation=45, ha='right')
    ax.set_yticks(range(len(tool_names)))
    ax.set_yticklabels(tool_names)
    
    # Add colorbar
    cbar = plt.colorbar(heatmap, ax=ax)
    cbar.set_label('Score', rotation=270, labelpad=15)
    
    # Add text annotations
    for i in range(len(tool_names)):
        for j in range(len(component_names)):
            text = ax.text(j, i, f'{heatmap_data[i][j]:.1f}',
                          ha="center", va="center", color="black", fontsize=8)
    
    ax.set_title('Component Scores for Top 20 Tools', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

create_quality_heatmap()

print("✓ All visualizations created successfully!")
print("\nVisualization Summary:")
print("  1. Tier Distribution - Shows how tools are distributed across quality tiers")
print("  2. Quality Grade Distribution - Shows the overall quality grades (A-F)")
print("  3. Component Score Box Plots - Shows the distribution of scores for each component")
print("  4. Score vs Error Count - Shows the relationship between quality and linter errors")
print("  5. Radar Chart - Compares average component scores across tiers")
print("  6. Quality Heatmap - Detailed view of component scores for top-performing tools")

## 9. Summarize and Report Frequently Missing or Malformed Attributes

Let's identify and visualize the most commonly missing or malformed metadata fields to highlight priority areas for improvement.

In [ ]:
# Comprehensive analysis of missing and malformed attributes
def analyze_field_completeness(tools_data):
    """Analyze completeness of all relevant fields."""
    field_analysis = {}
    
    # Define all important fields to check
    fields_to_check = {
        'basic_fields': ['name', 'description', 'homepage', 'biotoolsID', 'version'],
        'functional_fields': ['function', 'topic', 'toolType', 'operatingSystem'],
        'technical_fields': ['language', 'license', 'maturity', 'cost'],
        'documentation_fields': ['documentation', 'publication', 'publicationsPrimaryID'],
        'accessibility_fields': ['download', 'link', 'repository'],
        'community_fields': ['contact', 'credit', 'owner']
    }
    
    total_tools = len(tools_data)
    
    for category, fields in fields_to_check.items():
        field_analysis[category] = {}
        
        for field in fields:
            present_count = 0
            empty_count = 0
            malformed_count = 0
            
            for tool in tools_data:
                if field in tool:
                    value = tool[field]
                    if value:  # Not None or empty
                        if isinstance(value, list) and len(value) > 0:
                            present_count += 1
                        elif isinstance(value, str) and value.strip():
                            present_count += 1
                        elif not isinstance(value, (list, str)) and value:
                            present_count += 1
                        else:
                            empty_count += 1
                    else:
                        empty_count += 1
                else:
                    empty_count += 1
            
            missing_count = total_tools - present_count
            completeness_percentage = (present_count / total_tools) * 100
            
            field_analysis[category][field] = {
                'present': present_count,
                'missing': missing_count,
                'completeness_percentage': completeness_percentage,
                'priority_level': 'high' if completeness_percentage < 30 else 
                               'medium' if completeness_percentage < 60 else 'low'
            }
    
    return field_analysis

# Analyze field completeness
field_completeness = analyze_field_completeness(proteomics_tools)

# Create summary report
print("FIELD COMPLETENESS ANALYSIS")
print("=" * 50)

for category, fields in field_completeness.items():
    print(f"\n{category.upper().replace('_', ' ')}:")
    print("-" * 30)
    
    # Sort fields by completeness percentage
    sorted_fields = sorted(fields.items(), key=lambda x: x[1]['completeness_percentage'])
    
    for field, stats in sorted_fields:
        percentage = stats['completeness_percentage']
        priority = stats['priority_level']
        missing = stats['missing']
        
        # Color-code output based on priority
        priority_symbol = "🔴" if priority == 'high' else "🟡" if priority == 'medium' else "🟢"
        
        print(f"  {priority_symbol} {field:<20}: {percentage:5.1f}% complete ({missing:3d} missing)")

# Identify top priority fields for improvement
all_fields = []
for category, fields in field_completeness.items():
    for field, stats in fields.items():
        all_fields.append({
            'field': field,
            'category': category,
            'completeness': stats['completeness_percentage'],
            'missing': stats['missing'],
            'priority': stats['priority_level']
        })

# Sort by missing count (highest impact)
priority_fields = sorted(all_fields, key=lambda x: x['missing'], reverse=True)

print(f"\n\nTOP 10 PRIORITY FIELDS FOR IMPROVEMENT")
print("=" * 50)
print("(Ranked by number of tools missing this field)")
print()

for i, field_info in enumerate(priority_fields[:10], 1):
    field = field_info['field']
    missing = field_info['missing']
    completeness = field_info['completeness']
    category = field_info['category'].replace('_fields', '').title()
    
    print(f"{i:2d}. {field:<20} | {missing:3d} tools missing | {completeness:5.1f}% complete | {category}")

# Create visualization of missing fields
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Top 10 missing fields bar chart
top_missing = priority_fields[:10]
field_names = [f['field'] for f in top_missing]
missing_counts = [f['missing'] for f in top_missing]

bars = ax1.barh(field_names, missing_counts, color=['red' if f['priority'] == 'high' else 
                                                   'orange' if f['priority'] == 'medium' else 'green' 
                                                   for f in top_missing])
ax1.set_xlabel('Number of Tools Missing Field')
ax1.set_title('Top 10 Fields by Missing Count', fontweight='bold')
ax1.invert_yaxis()

# Add value labels on bars
for i, (bar, count) in enumerate(zip(bars, missing_counts)):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
             str(count), ha='left', va='center')

# Completeness percentage by category
category_completeness = {}
for category, fields in field_completeness.items():
    completeness_values = [stats['completeness_percentage'] for stats in fields.values()]
    category_completeness[category.replace('_fields', '').title()] = np.mean(completeness_values)

categories = list(category_completeness.keys())
completeness_avgs = list(category_completeness.values())

bars2 = ax2.bar(categories, completeness_avgs, color=colors[:len(categories)])
ax2.set_ylabel('Average Completeness (%)')
ax2.set_title('Average Completeness by Category', fontweight='bold')
ax2.set_ylim(0, 100)
ax2.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, avg in zip(bars2, completeness_avgs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{avg:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Generate improvement recommendations
print(f"\n\nIMPROVEMENT RECOMMENDATIONS")
print("=" * 50)

high_priority_fields = [f for f in priority_fields if f['priority'] == 'high']
medium_priority_fields = [f for f in priority_fields if f['priority'] == 'medium']

print(f"\n🔴 HIGH PRIORITY (< 30% complete): {len(high_priority_fields)} fields")
for field_info in high_priority_fields[:5]:
    field = field_info['field']
    missing = field_info['missing']
    print(f"   • {field}: {missing} tools need this field")

print(f"\n🟡 MEDIUM PRIORITY (30-60% complete): {len(medium_priority_fields)} fields")
for field_info in medium_priority_fields[:5]:
    field = field_info['field']
    missing = field_info['missing']
    print(f"   • {field}: {missing} tools could benefit from this field")

# Calculate potential impact
total_missing_fields = sum(f['missing'] for f in all_fields)
high_priority_missing = sum(f['missing'] for f in high_priority_fields)

print(f"\n📊 IMPACT ANALYSIS:")
print(f"   • Total missing field instances: {total_missing_fields}")
print(f"   • High priority missing fields: {high_priority_missing} ({high_priority_missing/total_missing_fields*100:.1f}%)")
print(f"   • Average tool completeness: {np.mean([f['completeness'] for f in all_fields]):.1f}%")

print(f"\n💡 KEY RECOMMENDATIONS:")
print(f"   1. Focus on the top 5 missing fields to achieve maximum impact")
print(f"   2. Implement validation checks for high-priority fields")
print(f"   3. Provide guidance and examples for frequently missing fields")
print(f"   4. Consider making high-priority fields required in submission guidelines")

## 10. Draft Proposed Revisions to Tool Information Standards

Based on our empirical findings, we'll draft evidence-based recommendations for modifying and clarifying the Tool Information Standards.

In [ ]:
# Generate comprehensive report with proposed revisions
def generate_revision_report():
    """Generate proposed revisions based on analysis findings."""
    
    report = f"""
PROPOSED REVISIONS TO TOOL INFORMATION STANDARDS
================================================
Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d')}
Dataset: {len(proteomics_tools)} proteomics tools from bio.tools
Average Quality Score: {df_integrated['composite_score'].mean():.2f}/100

EXECUTIVE SUMMARY
-----------------
Our analysis of {len(proteomics_tools)} proteomics tools reveals significant opportunities for 
improving metadata quality in the bio.tools registry. Key findings include:

• Only {len(df_integrated[df_integrated['quality_grade'].isin(['A', 'B'])])}/{len(df_integrated)} tools ({len(df_integrated[df_integrated['quality_grade'].isin(['A', 'B'])])/len(df_integrated)*100:.1f}%) achieve high quality grades (A or B)
• {len(df_integrated[df_integrated['tier'] <= 2])}/{len(df_integrated)} tools ({len(df_integrated[df_integrated['tier'] <= 2])/len(df_integrated)*100:.1f}%) are in the lowest quality tiers (1-2)
• Documentation and community information are the weakest areas overall

PRIORITY RECOMMENDATIONS
------------------------

1. ENHANCE REQUIRED FIELD SPECIFICATIONS
   Current Issue: {len([f for f in all_fields if f['priority'] == 'high'])} fields have less than 30% completion rate
   
   Recommendation: Elevate the following fields to "required" status:
   """
    
    # Add top missing fields
    top_5_missing = priority_fields[:5]
    for i, field_info in enumerate(top_5_missing, 1):
        report += f"\n   {i}. {field_info['field']}: Currently missing in {field_info['missing']} tools ({100-field_info['completeness']:.1f}% missing)"
    
    report += f"""

2. RESTRUCTURE TIER DEFINITIONS
   Current Issue: Tools cluster in lower tiers with unclear progression paths
   
   Proposed Tier Structure:
   • Tier 1 (Essential): Name, description, homepage, function, topic
   • Tier 2 (Functional): + toolType, operation details, basic documentation
   • Tier 3 (Technical): + language, license, version, operating system
   • Tier 4 (Community): + publications, contact information, detailed documentation
   • Tier 5 (Exemplary): + download links, repository, comprehensive metadata

3. STRENGTHEN DOCUMENTATION REQUIREMENTS
   Current Issue: Only {field_completeness['documentation_fields']['documentation']['completeness_percentage']:.1f}% of tools have documentation
   
   Recommendations:
   • Make at least one documentation link mandatory for Tier 2+
   • Provide templates for common documentation types
   • Encourage links to user guides, API documentation, and tutorials

4. IMPROVE COMMUNITY INFORMATION STANDARDS
   Current Issue: Contact information present in only {field_completeness['community_fields']['contact']['completeness_percentage']:.1f}% of tools
   
   Recommendations:
   • Require maintainer contact for all tools
   • Encourage ORCID IDs for contributors
   • Standardize credit attribution format

5. ENHANCE TECHNICAL METADATA GUIDANCE
   Current Issue: License information missing in {field_completeness['technical_fields']['license']['missing']} tools
   
   Recommendations:
   • Provide license selection wizard
   • Make license field required for publicly available tools
   • Standardize version numbering guidance

IMPLEMENTATION ROADMAP
---------------------

Phase 1 (0-3 months): Update submission guidelines
• Revise required field definitions
• Create field completion checklists
• Develop validation rules for high-priority fields

Phase 2 (3-6 months): Tool enhancement program
• Contact owners of low-tier tools for metadata improvement
• Provide automated suggestions based on missing fields
• Implement progressive disclosure in submission forms

Phase 3 (6-12 months): Community engagement
• Launch metadata quality campaigns
• Recognize high-quality annotations
• Provide training workshops for tool developers

EXPECTED OUTCOMES
----------------
Implementation of these recommendations could:
• Increase average quality score from {df_integrated['composite_score'].mean():.1f} to an estimated 75+ points
• Move {len(df_integrated[df_integrated['tier'] <= 2])} tools from Tier 1-2 to higher tiers
• Improve overall registry usability and discoverability

VALIDATION METRICS
-----------------
Success should be measured by:
• Reduction in missing high-priority fields by 50% within 6 months
• Increase in Tier 4-5 tools by 30% within 1 year
• Improved user satisfaction scores for tool discovery
• Reduced linter error rates across the registry

"""
    
    return report

# Generate and display the revision report
revision_report = generate_revision_report()
print(revision_report)

# Save comprehensive results to file
comprehensive_results = {
    'analysis_metadata': {
        'analysis_date': pd.Timestamp.now().isoformat(),
        'dataset_size': len(proteomics_tools),
        'focus_domain': 'proteomics'
    },
    'quality_assessment': {
        'average_composite_score': df_integrated['composite_score'].mean(),
        'score_distribution': df_integrated['composite_score'].describe().to_dict(),
        'tier_distribution': df_integrated['tier'].value_counts().to_dict(),
        'grade_distribution': df_integrated['quality_grade'].value_counts().to_dict()
    },
    'field_completeness': field_completeness,
    'top_priority_fields': [f for f in priority_fields[:10]],
    'linter_analysis': {
        'total_errors': sum(r['error_count'] for r in linter_results),
        'total_warnings': sum(r['warning_count'] for r in linter_results),
        'most_common_errors': dict(Counter(all_errors).most_common(10)),
        'most_common_warnings': dict(Counter(all_warnings).most_common(10))
    },
    'recommendations': {
        'required_field_additions': [f['field'] for f in priority_fields[:5] if f['priority'] == 'high'],
        'tier_restructuring': True,
        'documentation_enhancement': True,
        'community_standards': True
    }
}

# Save to JSON file for further processing
results_file = RESULTS_DIR / 'comprehensive_quality_analysis.json'
with open(results_file, 'w') as f:
    json.dump(comprehensive_results, f, indent=2, default=str)

print(f"\n✅ ANALYSIS COMPLETE!")
print(f"📊 Comprehensive results saved to: {results_file}")
print(f"📈 {len(proteomics_tools)} tools analyzed across {len(all_fields)} metadata fields")
print(f"🎯 {len([f for f in all_fields if f['priority'] == 'high'])} high-priority improvement areas identified")
print(f"📋 Evidence-based recommendations generated for Tool Information Standards revision")

# Final summary statistics
print(f"\n" + "="*60)
print(f"FINAL SUMMARY STATISTICS")
print(f"="*60)
print(f"Dataset: {len(proteomics_tools)} proteomics tools")
print(f"Average Quality Score: {df_integrated['composite_score'].mean():.2f}/100")
print(f"Tools needing improvement: {len(df_integrated[df_integrated['composite_score'] < 50])}/{len(df_integrated)} ({len(df_integrated[df_integrated['composite_score'] < 50])/len(df_integrated)*100:.1f}%)")
print(f"High-quality tools (Grade A-B): {len(df_integrated[df_integrated['quality_grade'].isin(['A', 'B'])])}/{len(df_integrated)} ({len(df_integrated[df_integrated['quality_grade'].isin(['A', 'B'])])/len(df_integrated)*100:.1f}%)")
print(f"Fields analyzed: {len(all_fields)} across 6 categories")
print(f"High-priority missing fields: {len([f for f in all_fields if f['priority'] == 'high'])}")
print(f"Potential for improvement: High - Many tools could advance tiers with focused metadata enhancement")
print(f"="*60)

In [ ]:
# Core libraries
import requests
import json
import pandas as pd
import numpy as np
from datetime import datetime
import logging
import time
import warnings
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import Counter, defaultdict

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Schema validation
import jsonschema
import yaml

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

# Constants
BIO_TOOLS_API_BASE = "https://bio.tools/api/tool/"
SAMPLE_LIMIT = 50  # For demonstration purposes
TIER_COLORS = {
    1: '#ff4d4d',  # Red - Minimal
    2: '#ff9933',  # Orange - Basic  
    3: '#ffcc00',  # Yellow - Moderate
    4: '#66cc00',  # Light Green - Good
    5: '#00cc66'   # Green - Excellent
}

print("✅ Libraries imported successfully!")
print(f"📊 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Load configuration
config_path = "../config/scoring_config.yaml"
try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("✅ Configuration loaded successfully!")
    print(f"📋 Scoring weights: {config['scoring']['weights']}")
    print(f"🎯 Tier thresholds: {config['scoring']['tiers']}")
except Exception as e:
    print(f"⚠️ Could not load config: {e}")
    # Use default configuration
    config = {
        'scoring': {
            'weights': {
                'basic_info': 20, 'detailed_description': 15, 'technical_details': 25,
                'documentation': 20, 'accessibility': 10, 'community': 10
            },
            'tiers': {
                'tier_1': [0, 20], 'tier_2': [21, 40], 'tier_3': [41, 60],
                'tier_4': [61, 80], 'tier_5': [81, 100]
            }
        }
    }

## 2. Retrieve Tool Metadata from bio.tools API

We'll create functions to fetch tool entries from the bio.tools API, with options to analyze specific collections or search for tools by topic.

In [ ]:
class BioToolsAPIClient:
    """Simple API client for bio.tools registry."""
    
    def __init__(self, base_url=BIO_TOOLS_API_BASE, timeout=30):
        self.base_url = base_url
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'bio.tools-quality-evaluation/1.0',
            'Content-Type': 'application/json'
        })
    
    def get_tools_by_topic(self, topic: str, limit: int = 50) -> List[Dict]:
        """Retrieve tools by topic."""
        tools = []
        page = 1
        
        while len(tools) < limit:
            params = {'topic': topic, 'page': page, 'page_size': min(100, limit - len(tools))}
            
            try:
                response = self.session.get(self.base_url, params=params, timeout=self.timeout)
                response.raise_for_status()
                data = response.json()
                
                if 'list' not in data or not data['list']:
                    break
                
                tools.extend(data['list'])
                page += 1
                time.sleep(1)  # Be respectful to the API
                
            except Exception as e:
                logger.error(f"Error fetching tools: {e}")
                break
        
        return tools[:limit]
    
    def search_tools(self, query: str, limit: int = 50) -> List[Dict]:
        """Search for tools using a query string."""
        tools = []
        page = 1
        
        while len(tools) < limit:
            params = {'q': query, 'page': page, 'page_size': min(100, limit - len(tools))}
            
            try:
                response = self.session.get(self.base_url, params=params, timeout=self.timeout)
                response.raise_for_status()
                data = response.json()
                
                if 'list' not in data or not data['list']:
                    break
                
                tools.extend(data['list'])
                page += 1
                time.sleep(1)
                
            except Exception as e:
                logger.error(f"Error searching tools: {e}")
                break
        
        return tools[:limit]

# Initialize API client
api_client = BioToolsAPIClient()
print("✅ API client initialized!")

In [ ]:
# Collect sample tools for analysis
print("🔍 Collecting proteomics tools...")
proteomics_tools = api_client.get_tools_by_topic("Proteomics", limit=SAMPLE_LIMIT)

print("🔍 Collecting genomics tools...")  
genomics_tools = api_client.get_tools_by_topic("Genomics", limit=SAMPLE_LIMIT)

# Combine for comprehensive analysis
all_tools = proteomics_tools + genomics_tools

print(f"📊 Collected {len(proteomics_tools)} proteomics tools")
print(f"📊 Collected {len(genomics_tools)} genomics tools") 
print(f"📊 Total tools for analysis: {len(all_tools)}")

# Display sample tool structure
if all_tools:
    print("\n📋 Sample tool structure:")
    sample_tool = all_tools[0]
    print(f"Tool ID: {sample_tool.get('biotoolsID', 'N/A')}")
    print(f"Name: {sample_tool.get('name', 'N/A')}")
    print(f"Description: {sample_tool.get('description', 'N/A')[:100]}...")
    print(f"Fields present: {len(sample_tool.keys())}")
    print(f"Top-level fields: {list(sample_tool.keys())[:10]}")
else:
    print("⚠️ No tools collected - using demo data for analysis")

## 3. Validate Metadata Against biotoolsSchema

We'll implement basic schema validation to check metadata compliance and identify structural issues.

In [ ]:
def validate_tool_structure(tool_data: Dict) -> Dict[str, Any]:
    """Basic validation of tool structure and required fields."""
    
    validation_result = {
        'is_valid': True,
        'errors': [],
        'warnings': [],
        'completeness_score': 0
    }
    
    # Define essential fields based on Tool Information Standards
    essential_fields = ['biotoolsID', 'name', 'description', 'homepage']
    important_fields = ['function', 'topic', 'toolType', 'language', 'license']
    optional_fields = ['documentation', 'publication', 'contact', 'download']
    
    # Check essential fields
    missing_essential = []
    for field in essential_fields:
        if field not in tool_data or not tool_data[field]:
            missing_essential.append(field)
            validation_result['errors'].append(f"Missing essential field: {field}")
    
    # Check important fields
    missing_important = []
    for field in important_fields:
        if field not in tool_data or not tool_data[field]:
            missing_important.append(field)
            validation_result['warnings'].append(f"Missing important field: {field}")
    
    # Calculate basic completeness score
    total_fields = len(essential_fields) + len(important_fields) + len(optional_fields)
    present_fields = sum(1 for field in essential_fields + important_fields + optional_fields 
                        if field in tool_data and tool_data[field])
    
    validation_result['completeness_score'] = (present_fields / total_fields) * 100
    validation_result['is_valid'] = len(missing_essential) == 0
    
    return validation_result

# Validate all collected tools
validation_results = []
for tool in all_tools:
    result = validate_tool_structure(tool)
    result['biotoolsID'] = tool.get('biotoolsID', 'unknown')
    result['name'] = tool.get('name', 'unknown')
    validation_results.append(result)

# Summary statistics
valid_tools = sum(1 for r in validation_results if r['is_valid'])
avg_completeness = np.mean([r['completeness_score'] for r in validation_results])

print(f"📊 Validation Summary:")
print(f"   ✅ Valid tools: {valid_tools}/{len(validation_results)} ({valid_tools/len(validation_results)*100:.1f}%)")
print(f"   📈 Average completeness: {avg_completeness:.1f}%")
print(f"   ⚠️ Tools with warnings: {sum(1 for r in validation_results if r['warnings'])}")

# Show most common missing fields
all_errors = []
all_warnings = []
for result in validation_results:
    all_errors.extend(result['errors'])
    all_warnings.extend(result['warnings'])

print(f"\n🔍 Most common issues:")
if all_errors:
    error_counts = Counter(all_errors)
    for error, count in error_counts.most_common(5):
        print(f"   🚫 {error}: {count} tools")

if all_warnings:
    warning_counts = Counter(all_warnings)
    for warning, count in warning_counts.most_common(5):
        print(f"   ⚠️ {warning}: {count} tools")

## 4. Implement Tier-Based Scoring System

Now we'll implement the comprehensive tier-based scoring system that maps Tool Information Standards to quantitative metrics.

In [ ]:
class CompletenessScorer:
    """Comprehensive scorer for bio.tools annotation quality."""
    
    def __init__(self, config):
        self.config = config
        self.weights = config['scoring']['weights']
        self.tiers = config['scoring']['tiers']
    
    def is_empty(self, value):
        """Check if a field value is considered empty."""
        if value is None:
            return True
        if isinstance(value, str) and not value.strip():
            return True
        if isinstance(value, (list, dict)) and not value:
            return True
        return False
    
    def score_basic_info(self, tool_data: Dict) -> Tuple[float, Dict]:
        """Score basic information completeness."""
        fields = ['name', 'description', 'homepage', 'biotoolsID', 'version']
        max_score = self.weights['basic_info']
        
        present_count = sum(1 for field in fields 
                           if field in tool_data and not self.is_empty(tool_data[field]))
        
        score = (present_count / len(fields)) * max_score
        
        return score, {
            'max_score': max_score,
            'present_count': present_count,
            'total_fields': len(fields),
            'completeness_ratio': present_count / len(fields)
        }
    
    def score_detailed_description(self, tool_data: Dict) -> Tuple[float, Dict]:
        """Score detailed description completeness."""
        max_score = self.weights['detailed_description']
        
        function_score = 0
        topic_score = 0
        operation_score = 0
        
        # Score functions
        if 'function' in tool_data and tool_data['function']:
            function_score = min(5, len(tool_data['function']) * 2)
        
        # Score topics
        if 'topic' in tool_data and tool_data['topic']:
            topic_score = min(5, len(tool_data['topic']) * 2)
        
        # Score operations
        operations = set()
        if 'function' in tool_data:
            for func in tool_data['function']:
                if 'operation' in func:
                    for op in func['operation']:
                        if isinstance(op, dict) and 'term' in op:
                            operations.add(op['term'])
        operation_score = min(5, len(operations))
        
        total_score = (function_score + topic_score + operation_score) / 15 * max_score
        
        return total_score, {
            'max_score': max_score,
            'function_score': function_score,
            'topic_score': topic_score,
            'operation_score': operation_score,
            'operation_count': len(operations)
        }
    
    def score_technical_details(self, tool_data: Dict) -> Tuple[float, Dict]:
        """Score technical details completeness."""
        fields = ['toolType', 'language', 'operatingSystem', 'license', 'maturity']
        max_score = self.weights['technical_details']
        
        present_count = sum(1 for field in fields 
                           if field in tool_data and not self.is_empty(tool_data[field]))
        
        score = (present_count / len(fields)) * max_score
        
        return score, {
            'max_score': max_score,
            'present_count': present_count,
            'total_fields': len(fields),
            'completeness_ratio': present_count / len(fields)
        }
    
    def score_documentation(self, tool_data: Dict) -> Tuple[float, Dict]:
        """Score documentation completeness."""
        max_score = self.weights['documentation']
        
        doc_score = 0
        pub_score = 0
        
        # Score documentation
        if 'documentation' in tool_data and tool_data['documentation']:
            doc_score = min(6, len(tool_data['documentation']) * 2)
        
        # Score publications
        if 'publication' in tool_data and tool_data['publication']:
            pub_score = min(8, len(tool_data['publication']) * 4)
        
        # Bonus for primary publication
        if 'publicationsPrimaryID' in tool_data and tool_data['publicationsPrimaryID']:
            pub_score = max(pub_score, 6)
        
        total_score = (doc_score + pub_score) / 14 * max_score
        
        return total_score, {
            'max_score': max_score,
            'documentation_score': doc_score,
            'publication_score': pub_score,
            'doc_count': len(tool_data.get('documentation', [])),
            'pub_count': len(tool_data.get('publication', []))
        }
    
    def score_accessibility(self, tool_data: Dict) -> Tuple[float, Dict]:
        """Score accessibility completeness."""
        max_score = self.weights['accessibility']
        
        download_score = 0
        link_score = 0
        
        if 'download' in tool_data and tool_data['download']:
            download_score = min(5, len(tool_data['download']) * 2)
        
        if 'link' in tool_data and tool_data['link']:
            link_score = min(5, len(tool_data['link']))
        
        total_score = (download_score + link_score) / 10 * max_score
        
        return total_score, {
            'max_score': max_score,
            'download_score': download_score,
            'link_score': link_score,
            'download_count': len(tool_data.get('download', [])),
            'link_count': len(tool_data.get('link', []))
        }
    
    def score_community(self, tool_data: Dict) -> Tuple[float, Dict]:
        """Score community information completeness."""
        max_score = self.weights['community']
        
        credit_score = 0
        contact_score = 0
        
        if 'credit' in tool_data and tool_data['credit']:
            credit_score = min(4, len(tool_data['credit']) * 2)
        
        if 'contact' in tool_data and tool_data['contact']:
            contact_score = min(6, len(tool_data['contact']) * 3)
        
        total_score = (credit_score + contact_score) / 10 * max_score
        
        return total_score, {
            'max_score': max_score,
            'credit_score': credit_score,
            'contact_score': contact_score,
            'credit_count': len(tool_data.get('credit', [])),
            'contact_count': len(tool_data.get('contact', []))
        }
    
    def determine_tier(self, score: float) -> int:
        """Determine tier based on score."""
        for tier_name, (min_score, max_score) in self.tiers.items():
            if min_score <= score <= max_score:
                return int(tier_name.split('_')[1])
        return 1
    
    def score_tool(self, tool_data: Dict) -> Dict[str, Any]:
        """Calculate comprehensive score for a tool."""
        # Calculate individual scores
        basic_score, basic_details = self.score_basic_info(tool_data)
        description_score, description_details = self.score_detailed_description(tool_data)
        technical_score, technical_details = self.score_technical_details(tool_data)
        documentation_score, documentation_details = self.score_documentation(tool_data)
        accessibility_score, accessibility_details = self.score_accessibility(tool_data)
        community_score, community_details = self.score_community(tool_data)
        
        # Calculate total score
        total_score = (basic_score + description_score + technical_score + 
                      documentation_score + accessibility_score + community_score)
        
        # Determine tier
        tier = self.determine_tier(total_score)
        
        return {
            'biotoolsID': tool_data.get('biotoolsID', 'unknown'),
            'name': tool_data.get('name', 'unknown'),
            'total_score': round(total_score, 2),
            'tier': tier,
            'scores': {
                'basic_info': round(basic_score, 2),
                'detailed_description': round(description_score, 2),
                'technical_details': round(technical_score, 2),
                'documentation': round(documentation_score, 2),
                'accessibility': round(accessibility_score, 2),
                'community': round(community_score, 2)
            },
            'details': {
                'basic_info': basic_details,
                'detailed_description': description_details,
                'technical_details': technical_details,
                'documentation': documentation_details,
                'accessibility': accessibility_details,
                'community': community_details
            }
        }

# Initialize scorer and score all tools
scorer = CompletenessScorer(config)
scoring_results = [scorer.score_tool(tool) for tool in all_tools]

print(f"✅ Scored {len(scoring_results)} tools using comprehensive tier-based system!")

## 5. Analyze Completeness Patterns Across Tool Collections

Let's analyze the scoring results to identify patterns and trends across different tool collections and domains.

In [ ]:
# Convert results to DataFrame for easier analysis
df_results = pd.DataFrame([
    {
        'biotoolsID': result['biotoolsID'],
        'name': result['name'],
        'total_score': result['total_score'],
        'tier': result['tier'],
        'basic_info': result['scores']['basic_info'],
        'detailed_description': result['scores']['detailed_description'],
        'technical_details': result['scores']['technical_details'],
        'documentation': result['scores']['documentation'],
        'accessibility': result['scores']['accessibility'],
        'community': result['scores']['community'],
        'collection': 'proteomics' if result in scoring_results[:len(proteomics_tools)] else 'genomics'
    }
    for i, result in enumerate(scoring_results)
])

# Basic statistics
print("📊 BASIC STATISTICS")
print("=" * 50)
print(f"Total tools analyzed: {len(df_results)}")
print(f"Average score: {df_results['total_score'].mean():.2f}")
print(f"Median score: {df_results['total_score'].median():.2f}")
print(f"Standard deviation: {df_results['total_score'].std():.2f}")
print(f"Score range: {df_results['total_score'].min():.2f} - {df_results['total_score'].max():.2f}")

# Tier distribution
print(f"\n🎯 TIER DISTRIBUTION")
print("=" * 30)
tier_counts = df_results['tier'].value_counts().sort_index()
for tier, count in tier_counts.items():
    percentage = (count / len(df_results)) * 100
    print(f"Tier {tier}: {count} tools ({percentage:.1f}%)")

# Collection comparison
print(f"\n🔬 COLLECTION COMPARISON")
print("=" * 35)
collection_stats = df_results.groupby('collection').agg({
    'total_score': ['mean', 'median', 'std', 'count'],
    'tier': lambda x: x.value_counts().to_dict()
}).round(2)

for collection in df_results['collection'].unique():
    collection_data = df_results[df_results['collection'] == collection]
    print(f"\n{collection.upper()}:")
    print(f"  Tools: {len(collection_data)}")
    print(f"  Avg score: {collection_data['total_score'].mean():.2f}")
    print(f"  Median score: {collection_data['total_score'].median():.2f}")
    print(f"  Tier distribution: {collection_data['tier'].value_counts().sort_index().to_dict()}")

# Component analysis
print(f"\n⚙️ COMPONENT ANALYSIS")
print("=" * 30)
components = ['basic_info', 'detailed_description', 'technical_details', 
              'documentation', 'accessibility', 'community']

component_stats = df_results[components].describe()
print("\nComponent scores (mean ± std):")
for component in components:
    mean_score = df_results[component].mean()
    std_score = df_results[component].std()
    max_possible = config['scoring']['weights'][component]
    print(f"  {component}: {mean_score:.2f} ± {std_score:.2f} (max: {max_possible})")

# Find top and bottom performers
print(f"\n🏆 TOP 5 PERFORMERS")
print("=" * 25)
top_performers = df_results.nlargest(5, 'total_score')[['name', 'total_score', 'tier', 'collection']]
for _, tool in top_performers.iterrows():
    print(f"  {tool['name'][:40]:<40} | Score: {tool['total_score']:5.1f} | Tier: {tool['tier']} | {tool['collection']}")

print(f"\n⚠️ BOTTOM 5 PERFORMERS")
print("=" * 28)
bottom_performers = df_results.nsmallest(5, 'total_score')[['name', 'total_score', 'tier', 'collection']]
for _, tool in bottom_performers.iterrows():
    print(f"  {tool['name'][:40]:<40} | Score: {tool['total_score']:5.1f} | Tier: {tool['tier']} | {tool['collection']}")

# Identify most problematic areas
print(f"\n🔍 COMPONENT WEAKNESS ANALYSIS")
print("=" * 40)
for component in components:
    max_score = config['scoring']['weights'][component]
    weak_tools = df_results[df_results[component] < max_score * 0.3]  # Less than 30% of max
    print(f"  {component}: {len(weak_tools)} tools score <30% ({len(weak_tools)/len(df_results)*100:.1f}%)")

# Missing field analysis (simplified)
print(f"\n📋 COMPLETENESS PATTERNS")
print("=" * 35)
print("Based on scoring patterns, commonly missing attributes:")

# Analyze which components have lowest average scores
component_means = df_results[components].mean()
sorted_components = component_means.sort_values()

print("\nComponents ranked by completeness (lowest first):")
for component, avg_score in sorted_components.items():
    max_score = config['scoring']['weights'][component]
    completeness_pct = (avg_score / max_score) * 100
    print(f"  {component}: {completeness_pct:.1f}% complete on average")

## 6. Integrate and Parse bio.tools Linter Output

We'll simulate linter analysis to identify structural and syntactic issues in the metadata.

In [ ]:
class BioToolsLinterSimulator:
    """Simulates bio.tools linter functionality for demonstration."""
    
    def __init__(self):
        self.error_patterns = {
            'missing_required_field': ['biotoolsID', 'name', 'description', 'homepage'],
            'invalid_url': ['homepage', 'download', 'link', 'documentation'],
            'invalid_format': ['license', 'language', 'operatingSystem'],
            'empty_field': ['description', 'function', 'topic'],
            'invalid_doi': ['publication', 'publicationsPrimaryID']
        }
    
    def check_url_format(self, url: str) -> bool:
        """Simple URL validation."""
        if not url or not isinstance(url, str):
            return False
        return url.startswith(('http://', 'https://')) and '.' in url
    
    def check_doi_format(self, doi: str) -> bool:
        """Simple DOI validation."""
        if not doi or not isinstance(doi, str):
            return False
        return doi.startswith('10.') and '/' in doi
    
    def lint_tool(self, tool_data: Dict) -> Dict[str, Any]:
        """Perform linting analysis on a tool."""
        errors = []
        warnings = []
        
        # Check required fields
        for field in self.error_patterns['missing_required_field']:
            if field not in tool_data or not tool_data[field]:
                errors.append(f"Missing required field: {field}")
        
        # Check URL fields
        url_fields = ['homepage']
        for field in url_fields:
            if field in tool_data and tool_data[field]:
                if not self.check_url_format(tool_data[field]):
                    errors.append(f"Invalid URL format in {field}")
        
        # Check download links
        if 'download' in tool_data:
            for i, download in enumerate(tool_data['download']):
                if isinstance(download, dict) and 'url' in download:
                    if not self.check_url_format(download['url']):
                        errors.append(f"Invalid download URL at index {i}")
        
        # Check documentation links
        if 'documentation' in tool_data:
            for i, doc in enumerate(tool_data['documentation']):
                if isinstance(doc, dict) and 'url' in doc:
                    if not self.check_url_format(doc['url']):
                        warnings.append(f"Invalid documentation URL at index {i}")
        
        # Check publications
        if 'publication' in tool_data:
            for i, pub in enumerate(tool_data['publication']):
                if isinstance(pub, dict) and 'doi' in pub:
                    if not self.check_doi_format(pub['doi']):
                        warnings.append(f"Invalid DOI format in publication {i}")
        
        # Check for empty descriptions
        if 'description' in tool_data:
            desc = tool_data['description']
            if isinstance(desc, str) and len(desc.strip()) < 20:
                warnings.append("Description is too short (< 20 characters)")
        
        # Check function completeness
        if 'function' in tool_data:
            for i, func in enumerate(tool_data['function']):
                if isinstance(func, dict):
                    if not func.get('operation'):
                        warnings.append(f"Function {i} missing operation information")
                    if not func.get('input') and not func.get('output'):
                        warnings.append(f"Function {i} missing input/output specifications")
        
        # Calculate severity score
        error_weight = len(errors) * 3
        warning_weight = len(warnings) * 1
        severity_score = error_weight + warning_weight
        
        return {
            'biotoolsID': tool_data.get('biotoolsID', 'unknown'),
            'errors': errors,
            'warnings': warnings,
            'error_count': len(errors),
            'warning_count': len(warnings),
            'severity_score': severity_score,
            'lint_status': 'PASS' if len(errors) == 0 else 'FAIL'
        }

# Run linter simulation on all tools
linter = BioToolsLinterSimulator()
linter_results = [linter.lint_tool(tool) for tool in all_tools]

# Analyze linter results
print("🔍 LINTER ANALYSIS RESULTS")
print("=" * 40)

total_errors = sum(result['error_count'] for result in linter_results)
total_warnings = sum(result['warning_count'] for result in linter_results)
passing_tools = sum(1 for result in linter_results if result['lint_status'] == 'PASS')

print(f"Total tools analyzed: {len(linter_results)}")
print(f"Passing tools: {passing_tools}/{len(linter_results)} ({passing_tools/len(linter_results)*100:.1f}%)")
print(f"Total errors: {total_errors}")
print(f"Total warnings: {total_warnings}")
print(f"Average errors per tool: {total_errors/len(linter_results):.2f}")
print(f"Average warnings per tool: {total_warnings/len(linter_results):.2f}")

# Most common errors and warnings
all_errors = []
all_warnings = []
for result in linter_results:
    all_errors.extend(result['errors'])
    all_warnings.extend(result['warnings'])

print(f"\n🚫 MOST COMMON ERRORS:")
error_counts = Counter(all_errors)
for error, count in error_counts.most_common(10):
    print(f"  {error}: {count} occurrences")

print(f"\n⚠️ MOST COMMON WARNINGS:")
warning_counts = Counter(all_warnings)
for warning, count in warning_counts.most_common(10):
    print(f"  {warning}: {count} occurrences")

# Tools with highest severity scores
print(f"\n📊 TOOLS WITH HIGHEST SEVERITY SCORES:")
severity_df = pd.DataFrame(linter_results).sort_values('severity_score', ascending=False)
for _, tool in severity_df.head(5).iterrows():
    print(f"  {tool['biotoolsID']}: {tool['severity_score']} points ({tool['error_count']} errors, {tool['warning_count']} warnings)")

print(f"\n✅ Tools passing all linter checks: {passing_tools}")

## 7. Merge Completeness Scores with Linter Diagnostics

Now we'll combine the completeness scoring with linter diagnostics to create an integrated quality assessment.

In [ ]:
# Create integrated quality assessment
integrated_results = []

for i, (score_result, linter_result) in enumerate(zip(scoring_results, linter_results)):
    # Calculate adjusted quality score
    base_score = score_result['total_score']
    
    # Apply penalties for linter issues
    error_penalty = linter_result['error_count'] * 5  # 5 points per error
    warning_penalty = linter_result['warning_count'] * 2  # 2 points per warning
    
    adjusted_score = max(0, base_score - error_penalty - warning_penalty)
    
    # Determine overall quality grade
    if adjusted_score >= 80 and linter_result['error_count'] == 0:
        quality_grade = 'A'
    elif adjusted_score >= 60 and linter_result['error_count'] <= 1:
        quality_grade = 'B'
    elif adjusted_score >= 40 and linter_result['error_count'] <= 3:
        quality_grade = 'C'
    elif adjusted_score >= 20:
        quality_grade = 'D'
    else:
        quality_grade = 'F'
    
    # Generate quality recommendations
    recommendations = []
    
    # Based on scoring weaknesses
    if score_result['scores']['basic_info'] < 15:
        recommendations.append("Complete basic information (name, description, homepage)")
    if score_result['scores']['documentation'] < 10:
        recommendations.append("Add documentation and publication references")
    if score_result['scores']['technical_details'] < 15:
        recommendations.append("Specify technical details (language, OS, license)")
    if score_result['scores']['community'] < 5:
        recommendations.append("Add contact information and credits")
    
    # Based on linter issues
    if linter_result['error_count'] > 0:
        recommendations.append(f"Fix {linter_result['error_count']} critical errors")
    if linter_result['warning_count'] > 3:
        recommendations.append(f"Address {linter_result['warning_count']} warnings")
    
    integrated_result = {
        'biotoolsID': score_result['biotoolsID'],
        'name': score_result['name'],
        'base_score': base_score,
        'adjusted_score': adjusted_score,
        'tier': score_result['tier'],
        'quality_grade': quality_grade,
        'error_count': linter_result['error_count'],
        'warning_count': linter_result['warning_count'],
        'lint_status': linter_result['lint_status'],
        'component_scores': score_result['scores'],
        'recommendations': recommendations,
        'collection': 'proteomics' if i < len(proteomics_tools) else 'genomics'
    }
    
    integrated_results.append(integrated_result)

# Create DataFrame for analysis
df_integrated = pd.DataFrame(integrated_results)

# Summary statistics
print("🔗 INTEGRATED QUALITY ASSESSMENT")
print("=" * 50)
print(f"Total tools assessed: {len(df_integrated)}")
print(f"Average base score: {df_integrated['base_score'].mean():.2f}")
print(f"Average adjusted score: {df_integrated['adjusted_score'].mean():.2f}")
print(f"Average penalty: {(df_integrated['base_score'] - df_integrated['adjusted_score']).mean():.2f}")

# Quality grade distribution
print(f"\n📊 QUALITY GRADE DISTRIBUTION:")
grade_counts = df_integrated['quality_grade'].value_counts().sort_index()
for grade, count in grade_counts.items():
    percentage = (count / len(df_integrated)) * 100
    print(f"  Grade {grade}: {count} tools ({percentage:.1f}%)")

# Collection comparison
print(f"\n🔬 INTEGRATED SCORES BY COLLECTION:")
for collection in df_integrated['collection'].unique():
    collection_data = df_integrated[df_integrated['collection'] == collection]
    print(f"\n{collection.upper()}:")
    print(f"  Average base score: {collection_data['base_score'].mean():.2f}")
    print(f"  Average adjusted score: {collection_data['adjusted_score'].mean():.2f}")
    print(f"  Grade distribution: {collection_data['quality_grade'].value_counts().to_dict()}")
    print(f"  Average errors: {collection_data['error_count'].mean():.1f}")
    print(f"  Average warnings: {collection_data['warning_count'].mean():.1f}")

# Top performers (adjusted score)
print(f"\n🏆 TOP 5 PERFORMERS (ADJUSTED SCORES):")
top_adjusted = df_integrated.nlargest(5, 'adjusted_score')
for _, tool in top_adjusted.iterrows():
    print(f"  {tool['name'][:35]:<35} | Score: {tool['adjusted_score']:5.1f} | Grade: {tool['quality_grade']} | Errors: {tool['error_count']}")

# Most improved vs. most penalized
print(f"\n📈 MOST PENALIZED TOOLS (Base vs Adjusted):")
df_integrated['penalty'] = df_integrated['base_score'] - df_integrated['adjusted_score']
most_penalized = df_integrated.nlargest(5, 'penalty')
for _, tool in most_penalized.iterrows():
    print(f"  {tool['name'][:35]:<35} | Penalty: -{tool['penalty']:4.1f} | Errors: {tool['error_count']} | Warnings: {tool['warning_count']}")

# Tools with perfect linter scores
perfect_linter = df_integrated[(df_integrated['error_count'] == 0) & (df_integrated['warning_count'] == 0)]
print(f"\n✨ TOOLS WITH PERFECT LINTER SCORES: {len(perfect_linter)}/{len(df_integrated)} ({len(perfect_linter)/len(df_integrated)*100:.1f}%)")

# Common recommendations
all_recommendations = []
for result in integrated_results:
    all_recommendations.extend(result['recommendations'])

print(f"\n💡 MOST COMMON IMPROVEMENT RECOMMENDATIONS:")
rec_counts = Counter(all_recommendations)
for rec, count in rec_counts.most_common(10):
    print(f"  {rec}: {count} tools ({count/len(df_integrated)*100:.1f}%)")

## 8. Generate Visual Summaries (Radar Charts, Heatmaps)

Let's create comprehensive visualizations to summarize our findings and make them easily interpretable.

In [ ]:
# Set up the plotting environment
plt.rcParams['figure.figsize'] = (15, 10)

# 1. Tier Distribution Pie Chart
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Tier Distribution', 'Score Distribution', 
                   'Quality Grades', 'Collection Comparison'),
    specs=[[{'type': 'pie'}, {'type': 'histogram'}],
           [{'type': 'pie'}, {'type': 'bar'}]]
)

# Tier distribution
tier_counts = df_results['tier'].value_counts().sort_index()
fig.add_trace(
    go.Pie(
        labels=[f'Tier {t}' for t in tier_counts.index],
        values=tier_counts.values,
        marker_colors=[TIER_COLORS[t] for t in tier_counts.index],
        name="Tiers"
    ),
    row=1, col=1
)

# Score distribution
fig.add_trace(
    go.Histogram(
        x=df_results['total_score'],
        nbinsx=20,
        marker_color='lightblue',
        name="Scores"
    ),
    row=1, col=2
)

# Quality grades
grade_counts = df_integrated['quality_grade'].value_counts().sort_index()
fig.add_trace(
    go.Pie(
        labels=[f'Grade {g}' for g in grade_counts.index],
        values=grade_counts.values,
        name="Grades"
    ),
    row=2, col=1
)

# Collection comparison
collection_means = df_integrated.groupby('collection')['adjusted_score'].mean()
fig.add_trace(
    go.Bar(
        x=collection_means.index,
        y=collection_means.values,
        marker_color=['#FF6B6B', '#4ECDC4'],
        name="Collections"
    ),
    row=2, col=2
)

fig.update_layout(height=800, showlegend=False, title_text="bio.tools Quality Assessment Overview")
fig.show()

# 2. Component Radar Chart
components = ['basic_info', 'detailed_description', 'technical_details', 
              'documentation', 'accessibility', 'community']

# Calculate average scores by tier
fig_radar = go.Figure()

for tier in sorted(df_results['tier'].unique()):
    tier_data = df_results[df_results['tier'] == tier]
    
    if len(tier_data) > 0:
        avg_scores = [tier_data[comp].mean() for comp in components]
        
        fig_radar.add_trace(go.Scatterpolar(
            r=avg_scores,
            theta=components,
            fill='toself',
            name=f'Tier {tier}',
            line_color=TIER_COLORS[tier]
        ))

fig_radar.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, max([config['scoring']['weights'][comp] for comp in components])]
        )
    ),
    showlegend=True,
    title="Component Scores by Tier (Radar Chart)",
    height=600
)
fig_radar.show()

# 3. Heatmap of Component Scores by Collection
fig_heatmap = plt.figure(figsize=(12, 8))

# Prepare data for heatmap
heatmap_data = []
collections = df_results['collection'].unique()

for collection in collections:
    collection_data = df_results[df_results['collection'] == collection]
    avg_scores = [collection_data[comp].mean() for comp in components]
    heatmap_data.append(avg_scores)

heatmap_df = pd.DataFrame(heatmap_data, 
                         index=collections, 
                         columns=components)

sns.heatmap(heatmap_df, 
            annot=True, 
            fmt='.1f', 
            cmap='RdYlGn',
            center=10,
            cbar_kws={'label': 'Average Score'})

plt.title('Component Scores Heatmap by Collection')
plt.ylabel('Collection')
plt.xlabel('Components')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 4. Error and Warning Analysis
fig_errors = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Error Distribution', 'Warning Distribution')
)

# Error counts
error_counts = df_integrated['error_count'].value_counts().sort_index()
fig_errors.add_trace(
    go.Bar(x=error_counts.index, y=error_counts.values, 
           marker_color='red', name='Errors'),
    row=1, col=1
)

# Warning counts
warning_counts = df_integrated['warning_count'].value_counts().sort_index()
fig_errors.add_trace(
    go.Bar(x=warning_counts.index, y=warning_counts.values, 
           marker_color='orange', name='Warnings'),
    row=1, col=2
)

fig_errors.update_layout(height=400, showlegend=False, 
                        title_text="Linter Issues Distribution")
fig_errors.show()

# 5. Score Impact Analysis (Base vs Adjusted)
plt.figure(figsize=(12, 8))

# Scatter plot showing base vs adjusted scores
plt.subplot(2, 2, 1)
plt.scatter(df_integrated['base_score'], df_integrated['adjusted_score'], 
           c=df_integrated['error_count'], cmap='Reds', alpha=0.6)
plt.plot([0, 100], [0, 100], 'k--', alpha=0.5)
plt.xlabel('Base Score')
plt.ylabel('Adjusted Score')
plt.title('Base vs Adjusted Scores (colored by error count)')
plt.colorbar(label='Error Count')

# Distribution of penalties
plt.subplot(2, 2, 2)
penalties = df_integrated['base_score'] - df_integrated['adjusted_score']
plt.hist(penalties, bins=20, alpha=0.7, color='orange')
plt.xlabel('Score Penalty')
plt.ylabel('Number of Tools')
plt.title('Distribution of Score Penalties')

# Tier changes due to linter issues
plt.subplot(2, 2, 3)
# Calculate what tier tools would be in based on adjusted score
def score_to_tier(score):
    for tier_name, (min_score, max_score) in config['scoring']['tiers'].items():
        if min_score <= score <= max_score:
            return int(tier_name.split('_')[1])
    return 1

df_integrated['adjusted_tier'] = df_integrated['adjusted_score'].apply(score_to_tier)
tier_changes = (df_integrated['tier'] != df_integrated['adjusted_tier']).sum()

tier_change_counts = df_integrated.groupby(['tier', 'adjusted_tier']).size().unstack(fill_value=0)
sns.heatmap(tier_change_counts, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Adjusted Tier')
plt.ylabel('Original Tier')
plt.title(f'Tier Changes Due to Linter Issues ({tier_changes} tools affected)')

# Quality grade vs tier comparison
plt.subplot(2, 2, 4)
grade_tier_cross = pd.crosstab(df_integrated['tier'], df_integrated['quality_grade'])
grade_tier_cross.plot(kind='bar', stacked=True)
plt.xlabel('Original Tier')
plt.ylabel('Number of Tools')
plt.title('Quality Grade Distribution by Tier')
plt.legend(title='Quality Grade')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print("📊 All visualizations generated!")
print(f"📈 Key findings:")
print(f"   • {tier_changes} tools changed tiers due to linter issues")
print(f"   • Average penalty: {penalties.mean():.1f} points")
print(f"   • {len(perfect_linter)} tools have perfect linter scores")
print(f"   • Most problematic component: {component_means.sort_values().index[0]}")

## 9. Summarize and Report Frequently Missing or Malformed Attributes

Let's create a comprehensive summary of the most critical quality gaps and improvement opportunities.

In [ ]:
# Generate comprehensive quality gap analysis
def analyze_field_completeness(tools_data, scoring_results):
    """Analyze completeness of individual fields across all tools."""
    
    field_analysis = {}
    
    # Define field categories and their importance
    field_categories = {
        'Essential': ['biotoolsID', 'name', 'description', 'homepage'],
        'Functional': ['function', 'topic', 'operation', 'toolType'],
        'Technical': ['language', 'operatingSystem', 'license', 'version'],
        'Documentation': ['documentation', 'publication', 'publicationsPrimaryID'],
        'Accessibility': ['download', 'link', 'repository'],
        'Community': ['contact', 'credit', 'owner']
    }
    
    for category, fields in field_categories.items():
        field_analysis[category] = {}
        
        for field in fields:
            present_count = 0
            total_count = len(tools_data)
            quality_issues = []
            
            for tool in tools_data:
                if field in tool and tool[field]:
                    # Check if field has content
                    value = tool[field]
                    if isinstance(value, str) and value.strip():
                        present_count += 1
                    elif isinstance(value, list) and value:
                        present_count += 1
                    elif isinstance(value, dict) and value:
                        present_count += 1
                    else:
                        quality_issues.append('empty_value')
                else:
                    quality_issues.append('missing_field')
            
            completeness_pct = (present_count / total_count) * 100
            
            field_analysis[category][field] = {
                'present_count': present_count,
                'total_count': total_count,
                'completeness_percentage': completeness_pct,
                'missing_count': total_count - present_count,
                'quality_issues': Counter(quality_issues)
            }
    
    return field_analysis

# Perform field completeness analysis
field_completeness = analyze_field_completeness(all_tools, scoring_results)

print("📊 COMPREHENSIVE FIELD COMPLETENESS ANALYSIS")
print("=" * 60)

# Summary by category
for category, fields in field_completeness.items():
    print(f"\n📋 {category.upper()} FIELDS:")
    print("-" * 40)
    
    # Sort fields by completeness percentage
    sorted_fields = sorted(fields.items(), key=lambda x: x[1]['completeness_percentage'])
    
    for field, data in sorted_fields:
        completeness = data['completeness_percentage']
        missing = data['missing_count']
        
        # Determine status emoji
        if completeness >= 80:
            status = "✅"
        elif completeness >= 60:
            status = "⚠️"
        else:
            status = "🚫"
        
        print(f"  {status} {field:<20} {completeness:5.1f}% complete ({missing:3d} missing)")

# Identify critical gaps
print(f"\n🚨 CRITICAL QUALITY GAPS (< 50% completeness):")
print("=" * 55)

critical_gaps = []
for category, fields in field_completeness.items():
    for field, data in fields.items():
        if data['completeness_percentage'] < 50:
            critical_gaps.append((field, data['completeness_percentage'], category))

critical_gaps.sort(key=lambda x: x[1])  # Sort by completeness percentage

for field, completeness, category in critical_gaps:
    print(f"  🔴 {field} ({category}): {completeness:.1f}% complete")

# Analyze patterns by tier
print(f"\n📈 COMPLETENESS PATTERNS BY TIER:")
print("=" * 40)

tier_field_analysis = {}
for tier in sorted(df_results['tier'].unique()):
    tier_tools = df_results[df_results['tier'] == tier]
    tier_indices = tier_tools.index.tolist()
    tier_tool_data = [all_tools[i] for i in tier_indices]
    
    # Calculate average completeness for this tier
    tier_completeness = {}
    for category, fields in field_completeness.items():
        category_completeness = []
        for field in fields:
            field_present = sum(1 for tool in tier_tool_data 
                              if field in tool and tool[field])
            field_completeness = (field_present / len(tier_tool_data)) * 100 if tier_tool_data else 0
            category_completeness.append(field_completeness)
        
        tier_completeness[category] = np.mean(category_completeness) if category_completeness else 0
    
    tier_field_analysis[tier] = tier_completeness
    
    print(f"\nTier {tier} ({len(tier_tool_data)} tools):")
    for category, avg_completeness in tier_completeness.items():
        print(f"  {category}: {avg_completeness:.1f}% average completeness")

# Generate priority recommendations
print(f"\n🎯 PRIORITY IMPROVEMENT RECOMMENDATIONS:")
print("=" * 50)

recommendations = []

# Based on critical gaps
if critical_gaps:
    worst_gap = critical_gaps[0]
    recommendations.append(f"Immediate: Address {worst_gap[0]} completeness ({worst_gap[1]:.1f}% complete)")

# Based on component scores
lowest_component = df_results[components].mean().sort_values().index[0]
recommendations.append(f"Focus area: Improve {lowest_component} annotations across all tools")

# Based on linter results
if total_errors > 0:
    recommendations.append(f"Technical: Fix {total_errors} linter errors across {len(linter_results)} tools")

# Based on tier distribution
tier_1_count = len(df_results[df_results['tier'] == 1])
if tier_1_count > len(df_results) * 0.3:  # More than 30% in lowest tier
    recommendations.append(f"Strategic: Upgrade {tier_1_count} Tier 1 tools to higher tiers")

for i, rec in enumerate(recommendations, 1):
    print(f"  {i}. {rec}")

# Create summary report
print(f"\n📄 EXECUTIVE SUMMARY:")
print("=" * 30)
print(f"Tools analyzed: {len(all_tools)}")
print(f"Average quality score: {df_results['total_score'].mean():.1f}/100")
print(f"Tools passing linter: {passing_tools}/{len(linter_results)} ({passing_tools/len(linter_results)*100:.1f}%)")
print(f"Critical quality gaps: {len(critical_gaps)} fields")
print(f"Most problematic area: {lowest_component}")
print(f"Potential for tier upgrades: {len(df_integrated[df_integrated['penalty'] > 10])} tools")

# Quality impact estimation
potential_improvement = sum(result['penalty'] for result in 
                          [r for r in integrated_results if r['error_count'] > 0])
print(f"Potential score improvement: +{potential_improvement:.0f} points total")

print(f"\n✨ Analysis complete! Key insights:")
print(f"   • Most critical gap: {critical_gaps[0][0] if critical_gaps else 'None'}")
print(f"   • Best performing collection: {collection_means.sort_values().index[-1]}")
print(f"   • Tools ready for tier upgrade: {len(df_integrated[df_integrated['penalty'] < 5])}")

## 10. Draft Proposed Revisions to Tool Information Standards

Based on our empirical findings, let's draft evidence-based recommendations for improving the Tool Information Standards.

In [ ]:
# Generate evidence-based recommendations for Tool Information Standards
def generate_standards_recommendations(field_completeness, scoring_results, linter_results, integrated_results):
    """Generate recommendations based on empirical analysis."""
    
    recommendations = {
        'critical_revisions': [],
        'priority_improvements': [],
        'new_requirements': [],
        'guidance_updates': [],
        'validation_enhancements': []
    }
    
    # Analyze field completeness to identify critical issues
    all_fields = []
    for category, fields in field_completeness.items():
        for field, data in fields.items():
            all_fields.append((field, data['completeness_percentage'], category))
    
    # Sort by completeness
    all_fields.sort(key=lambda x: x[1])
    
    # Critical revisions for fields with <30% completeness
    critical_fields = [f for f in all_fields if f[1] < 30]
    if critical_fields:
        recommendations['critical_revisions'].append({
            'issue': 'Severely under-utilized fields',
            'fields': [f[0] for f in critical_fields[:5]],
            'recommendation': 'Consider removing, merging, or providing better guidance for these fields',
            'evidence': f'{len(critical_fields)} fields have <30% completeness'
        })
    
    # Priority improvements for fields with 30-60% completeness
    priority_fields = [f for f in all_fields if 30 <= f[1] < 60]
    if priority_fields:
        recommendations['priority_improvements'].append({
            'issue': 'Moderate completeness fields need attention',
            'fields': [f[0] for f in priority_fields[:5]],
            'recommendation': 'Provide enhanced guidance, examples, and validation for these fields',
            'evidence': f'{len(priority_fields)} fields have 30-60% completeness'
        })
    
    # Analyze linter results for validation improvements
    common_errors = Counter()
    for result in linter_results:
        common_errors.update(result['errors'])
    
    if common_errors:
        top_errors = common_errors.most_common(3)
        recommendations['validation_enhancements'].append({
            'issue': 'Common validation errors',
            'errors': [error for error, count in top_errors],
            'recommendation': 'Implement better client-side validation and clearer format requirements',
            'evidence': f'Top error affects {top_errors[0][1]} tools'
        })
    
    # Analyze tier distribution for new requirements
    tier_dist = Counter(result['tier'] for result in scoring_results)
    tier_1_percentage = (tier_dist[1] / len(scoring_results)) * 100
    
    if tier_1_percentage > 40:  # More than 40% in lowest tier
        recommendations['new_requirements'].append({
            'issue': 'Too many tools in lowest quality tier',
            'recommendation': 'Establish minimum requirements for tool registration',
            'evidence': f'{tier_1_percentage:.1f}% of tools are in Tier 1 (minimal annotation)',
            'suggested_minimums': ['name', 'description', 'homepage', 'function', 'topic']
        })
    
    # Component-specific recommendations
    component_scores = {comp: np.mean([r['scores'][comp] for r in scoring_results]) 
                       for comp in components}
    worst_component = min(component_scores, key=component_scores.get)
    
    recommendations['guidance_updates'].append({
        'issue': f'Weak performance in {worst_component}',
        'recommendation': f'Develop comprehensive guidance and examples for {worst_component} annotation',
        'evidence': f'Average score: {component_scores[worst_component]:.1f}/{config["scoring"]["weights"][worst_component]}'
    })
    
    return recommendations

# Generate recommendations
standards_recommendations = generate_standards_recommendations(
    field_completeness, scoring_results, linter_results, integrated_results
)

print("🔬 PROPOSED REVISIONS TO TOOL INFORMATION STANDARDS")
print("=" * 70)
print("Based on empirical analysis of annotation quality patterns\n")

# Display recommendations by category
for category, recs in standards_recommendations.items():
    if recs:  # Only show categories that have recommendations
        print(f"📋 {category.upper().replace('_', ' ')}:")
        print("-" * 50)
        
        for i, rec in enumerate(recs, 1):
            print(f"  {i}. {rec['issue']}")
            print(f"     Recommendation: {rec['recommendation']}")
            print(f"     Evidence: {rec['evidence']}")
            
            if 'fields' in rec:
                print(f"     Affected fields: {', '.join(rec['fields'])}")
            if 'errors' in rec:
                print(f"     Common errors: {', '.join(rec['errors'])}")
            if 'suggested_minimums' in rec:
                print(f"     Suggested minimums: {', '.join(rec['suggested_minimums'])}")
            print()

# Draft specific standard revisions
print(f"\n📝 DRAFT STANDARD REVISIONS:")
print("=" * 40)

print(f"""
**PROPOSED TOOL INFORMATION STANDARDS v2.0**

1. **MINIMUM REGISTRATION REQUIREMENTS** (NEW)
   - All tools must provide: name, description, homepage, at least one function, at least one topic
   - Rationale: {tier_dist[1]} tools ({tier_dist[1]/len(scoring_results)*100:.1f}%) currently lack basic annotation

2. **ENHANCED FIELD GUIDANCE**
   - Provide detailed examples for {worst_component} annotations
   - Mandate structured input/output specifications for function descriptions
   - Rationale: {worst_component} shows lowest completion rates

3. **VALIDATION IMPROVEMENTS**
   - Implement real-time URL validation for homepage, download, and documentation links
   - Add DOI format validation for publications
   - Provide format templates for common fields
   - Rationale: {total_errors} validation errors identified across {len(linter_results)} tools

4. **TIER-BASED GUIDELINES** (NEW)
   - Define progressive annotation requirements for quality tiers
   - Provide tier-specific checklists and guidance
   - Rationale: Clear quality progression encourages improvement

5. **COLLECTION-SPECIFIC GUIDANCE**
   - Develop domain-specific annotation guidelines (e.g., proteomics, genomics)
   - Provide field examples relevant to each scientific domain
   - Rationale: Quality varies significantly between collections

6. **AUTOMATED QUALITY ASSESSMENT** (NEW)
   - Implement automated scoring system for continuous quality monitoring
   - Provide quality badges/indicators for tools
   - Send improvement recommendations to tool maintainers
   - Rationale: Enable scalable quality improvement across the registry
""")

# Implementation roadmap
print(f"\n🗓️ IMPLEMENTATION ROADMAP:")
print("=" * 35)

roadmap = [
    ("Phase 1 (Immediate)", [
        "Deploy improved validation for new tool submissions",
        "Create field completion guidance documents",
        "Implement basic quality scoring"
    ]),
    ("Phase 2 (3-6 months)", [
        "Roll out tier-based guidelines",
        "Develop domain-specific guidance",
        "Launch quality improvement outreach"
    ]),
    ("Phase 3 (6-12 months)", [
        "Implement minimum registration requirements",
        "Deploy automated quality assessment system",
        "Evaluate impact and iterate"
    ])
]

for phase, tasks in roadmap:
    print(f"\n{phase}:")
    for task in tasks:
        print(f"  • {task}")

# Success metrics
print(f"\n📊 SUCCESS METRICS:")
print("=" * 25)
print(f"""
Target improvements after implementation:
• Increase average quality score from {df_results['total_score'].mean():.1f} to >60
• Reduce Tier 1 tools from {tier_dist[1]} ({tier_dist[1]/len(scoring_results)*100:.1f}%) to <20%
• Achieve >90% linter pass rate (currently {passing_tools/len(linter_results)*100:.1f}%)
• Improve worst component ({worst_component}) scores by 50%
• Increase critical field completeness to >70% for all essential fields
""")

print(f"\n🎯 CONCLUSION:")
print("=" * 20)
print(f"""
This analysis of {len(all_tools)} tools reveals significant opportunities for improving
bio.tools annotation quality through evidence-based standards revision. The proposed
changes are designed to be:

✅ Data-driven: Based on empirical analysis of real annotation patterns
✅ Practical: Addressing the most common and impactful quality issues  
✅ Progressive: Allowing tools to improve incrementally through tiers
✅ Automated: Leveraging technology to scale quality improvement

Implementation of these recommendations should significantly enhance the utility
and reliability of the bio.tools registry for both end users and developers.
""")

# Save results summary
results_summary = {
    'analysis_date': datetime.now().isoformat(),
    'tools_analyzed': len(all_tools),
    'average_score': float(df_results['total_score'].mean()),
    'tier_distribution': dict(tier_dist),
    'critical_gaps': len(critical_gaps) if 'critical_gaps' in locals() else 0,
    'linter_pass_rate': float(passing_tools/len(linter_results)*100),
    'recommendations': standards_recommendations
}

print(f"\n💾 Analysis complete! Results saved to variables for further export.")
print(f"📈 Ready for implementation planning and stakeholder review.")

In [ ]:
# Core libraries
import requests
import json
import pandas as pd
import numpy as np
from datetime import datetime
import logging
import time
import warnings
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import Counter, defaultdict

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Schema validation
import jsonschema
import yaml

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

# Constants
BIO_TOOLS_API_BASE = "https://bio.tools/api/tool/"
SAMPLE_LIMIT = 50  # For demonstration purposes
TIER_COLORS = {
    1: '#ff4d4d',  # Red - Minimal
    2: '#ff9933',  # Orange - Basic  
    3: '#ffcc00',  # Yellow - Moderate
    4: '#66cc00',  # Light Green - Good
    5: '#00cc66'   # Green - Excellent
}

print("✅ Libraries imported successfully!")
print(f"📊 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Retrieve Tool Metadata from bio.tools API

We'll create functions to fetch tool entries from the bio.tools API, focusing on specific collections (e.g., proteomics) for targeted analysis.

In [ ]:
class BioToolsAPI:
    """Simple API client for bio.tools registry."""
    
    def __init__(self, base_url: str = BIO_TOOLS_API_BASE):
        self.base_url = base_url
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'bio.tools-quality-evaluation/1.0',
            'Content-Type': 'application/json'
        })
    
    def get_tools_by_collection(self, collection: str, limit: int = 100) -> List[Dict]:
        """Retrieve tools from a specific collection."""
        tools = []
        page = 1
        
        while len(tools) < limit:
            params = {
                'collection': collection,
                'page': page,
                'page_size': min(100, limit - len(tools))
            }
            
            try:
                response = self.session.get(self.base_url, params=params, timeout=30)
                response.raise_for_status()
                data = response.json()
                
                page_tools = data.get('list', [])
                if not page_tools:
                    break
                
                tools.extend(page_tools)
                logger.info(f"Retrieved {len(tools)} tools so far...")
                
                if len(page_tools) < params['page_size']:
                    break
                
                page += 1
                time.sleep(1)  # Be respectful to the API
                
            except Exception as e:
                logger.error(f"Failed to retrieve tools: {e}")
                break
        
        return tools[:limit]
    
    def get_tools_by_topic(self, topic: str, limit: int = 100) -> List[Dict]:
        """Retrieve tools by topic."""
        tools = []
        page = 1
        
        while len(tools) < limit:
            params = {
                'topic': topic,
                'page': page,
                'page_size': min(100, limit - len(tools))
            }
            
            try:
                response = self.session.get(self.base_url, params=params, timeout=30)
                response.raise_for_status()
                data = response.json()
                
                page_tools = data.get('list', [])
                if not page_tools:
                    break
                
                tools.extend(page_tools)
                logger.info(f"Retrieved {len(tools)} tools so far...")
                
                if len(page_tools) < params['page_size']:
                    break
                
                page += 1
                time.sleep(1)
                
            except Exception as e:
                logger.error(f"Failed to retrieve tools: {e}")
                break
        
        return tools[:limit]

# Initialize API client
api_client = BioToolsAPI()
print("✅ API client initialized successfully!")

In [ ]:
# Collect sample data for analysis
print("🔍 Collecting proteomics tools for analysis...")

# For demonstration purposes, we'll use fewer tools to avoid long API calls
# In a real analysis, you would use larger samples
proteomics_tools = api_client.get_tools_by_topic("Proteomics", limit=SAMPLE_LIMIT)

print(f"📦 Collected {len(proteomics_tools)} proteomics tools")

# Display basic information about the first few tools
if proteomics_tools:
    print("\n📋 Sample of collected tools:")
    for i, tool in enumerate(proteomics_tools[:3]):
        print(f"{i+1}. {tool.get('name', 'Unknown')} (ID: {tool.get('biotoolsID', 'N/A')})")
        print(f"   Description: {tool.get('description', 'No description')[:100]}...")
        print()
else:
    print("❌ No tools collected. Using sample data for demonstration.")
    # Fallback sample data
    proteomics_tools = [
        {
            "biotoolsID": "example_tool_1",
            "name": "Sample Proteomics Tool",
            "description": "A sample tool for proteomics analysis",
            "homepage": "https://example.com",
            "topic": [{"term": "Proteomics"}],
            "function": [{"operation": [{"term": "Protein identification"}]}]
        }
    ]

## 3. Validate Metadata Against biotoolsSchema

We'll implement JSON schema validation to check each tool's metadata for compliance with the biotoolsSchema. This helps identify structural issues in the data.

In [ ]:
class SchemaValidator:
    """Validator for bio.tools metadata using JSON schema."""
    
    def __init__(self):
        # For demonstration, we'll use a simplified schema structure
        # In practice, you would load the full biotoolsSchema
        self.basic_schema = {
            "type": "object",
            "required": ["biotoolsID", "name", "description"],
            "properties": {
                "biotoolsID": {"type": "string"},
                "name": {"type": "string"},
                "description": {"type": "string"},
                "homepage": {"type": "string", "format": "uri"},
                "topic": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "term": {"type": "string"}
                        }
                    }
                },
                "function": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "operation": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "term": {"type": "string"}
                                    }
                                }
                            }
                        }
                    }
                }
            }
        }
    
    def validate_tool(self, tool_data: Dict) -> Dict[str, Any]:
        """Validate a single tool against the schema."""
        result = {
            'is_valid': True,
            'errors': [],
            'warnings': []
        }
        
        try:
            jsonschema.validate(tool_data, self.basic_schema)
        except jsonschema.ValidationError as e:
            result['is_valid'] = False
            result['errors'].append({
                'message': e.message,
                'path': list(e.path),
                'invalid_value': e.instance
            })
        except Exception as e:
            result['errors'].append(f"Validation error: {str(e)}")
        
        return result
    
    def validate_tools(self, tools: List[Dict]) -> List[Dict[str, Any]]:
        """Validate multiple tools."""
        results = []
        for i, tool in enumerate(tools):
            try:
                validation_result = self.validate_tool(tool)
                validation_result['tool_index'] = i
                validation_result['biotoolsID'] = tool.get('biotoolsID', f'unknown_{i}')
                results.append(validation_result)
            except Exception as e:
                logger.error(f"Failed to validate tool {i}: {e}")
                results.append({
                    'tool_index': i,
                    'biotoolsID': tool.get('biotoolsID', f'unknown_{i}'),
                    'is_valid': False,
                    'errors': [str(e)]
                })
        
        return results

# Initialize validator
validator = SchemaValidator()

# Validate the collected tools
print("🔍 Validating tools against schema...")
validation_results = validator.validate_tools(proteomics_tools)

# Summary of validation results
valid_count = sum(1 for r in validation_results if r['is_valid'])
invalid_count = len(validation_results) - valid_count

print(f"✅ Valid tools: {valid_count}")
print(f"❌ Invalid tools: {invalid_count}")

if invalid_count > 0:
    print("\n🚨 Sample validation errors:")
    for result in validation_results[:3]:
        if not result['is_valid'] and result['errors']:
            print(f"- {result['biotoolsID']}: {result['errors'][0].get('message', 'Unknown error')}")

## 4. Implement Tier-Based Scoring System

We'll create a comprehensive scoring system that maps Tool Information Standards to a numerical rubric, assigning completeness scores and tiers (1–5) to each tool based on metadata richness.

In [ ]:
class QualityScorer:
    """Comprehensive scoring system for bio.tools annotation quality."""
    
    def __init__(self):
        # Scoring weights aligned with Tool Information Standards
        self.weights = {
            'basic_info': 20,        # Name, description, homepage, biotoolsID
            'detailed_description': 15,  # Function, topic, operation
            'technical_details': 25,    # Version, language, OS, license
            'documentation': 20,       # Documentation links, publications
            'accessibility': 10,       # Download links, repositories
            'community': 10           # Credits, contact information
        }
        
        # Tier thresholds
        self.tier_thresholds = {
            1: (0, 20),    # Minimal
            2: (21, 40),   # Basic
            3: (41, 60),   # Moderate
            4: (61, 80),   # Good
            5: (81, 100)   # Excellent
        }
    
    def _is_empty(self, value) -> bool:
        """Check if a value is considered empty."""
        if value is None:
            return True
        if isinstance(value, str) and not value.strip():
            return True
        if isinstance(value, (list, dict)) and not value:
            return True
        return False
    
    def score_basic_info(self, tool: Dict) -> Tuple[float, Dict]:
        """Score basic information completeness."""
        fields = ['name', 'description', 'homepage', 'biotoolsID']
        present = sum(1 for field in fields if not self._is_empty(tool.get(field)))
        score = (present / len(fields)) * self.weights['basic_info']
        
        details = {
            'present_fields': [f for f in fields if not self._is_empty(tool.get(f))],
            'missing_fields': [f for f in fields if self._is_empty(tool.get(f))],
            'completeness_ratio': present / len(fields)
        }
        
        return score, details
    
    def score_detailed_description(self, tool: Dict) -> Tuple[float, Dict]:
        """Score detailed description completeness."""
        max_score = self.weights['detailed_description']
        
        # Score functions
        functions = tool.get('function', [])
        function_score = min(5, len(functions)) if functions else 0
        
        # Score topics
        topics = tool.get('topic', [])
        topic_score = min(5, len(topics)) if topics else 0
        
        # Score operations (from functions)
        operations = []
        for func in functions:
            operations.extend(func.get('operation', []))
        operation_score = min(5, len(operations)) if operations else 0
        
        total_score = (function_score + topic_score + operation_score) / 15 * max_score
        
        details = {
            'function_count': len(functions),
            'topic_count': len(topics),
            'operation_count': len(operations),
            'function_score': function_score,
            'topic_score': topic_score,
            'operation_score': operation_score
        }
        
        return total_score, details
    
    def score_technical_details(self, tool: Dict) -> Tuple[float, Dict]:
        """Score technical details completeness."""
        fields = ['toolType', 'language', 'operatingSystem', 'license', 'version']
        present = sum(1 for field in fields if not self._is_empty(tool.get(field)))
        score = (present / len(fields)) * self.weights['technical_details']
        
        details = {
            'present_fields': [f for f in fields if not self._is_empty(tool.get(f))],
            'missing_fields': [f for f in fields if self._is_empty(tool.get(f))],
            'completeness_ratio': present / len(fields)
        }
        
        return score, details
    
    def score_documentation(self, tool: Dict) -> Tuple[float, Dict]:
        """Score documentation completeness."""
        max_score = self.weights['documentation']
        
        # Score documentation links
        docs = tool.get('documentation', [])
        doc_score = min(10, len(docs) * 3) if docs else 0
        
        # Score publications
        pubs = tool.get('publication', [])
        pub_score = min(10, len(pubs) * 5) if pubs else 0
        
        total_score = (doc_score + pub_score) / 20 * max_score
        
        details = {
            'documentation_count': len(docs),
            'publication_count': len(pubs),
            'doc_score': doc_score,
            'pub_score': pub_score
        }
        
        return total_score, details
    
    def score_accessibility(self, tool: Dict) -> Tuple[float, Dict]:
        """Score accessibility completeness."""
        max_score = self.weights['accessibility']
        
        downloads = tool.get('download', [])
        links = tool.get('link', [])
        repos = tool.get('repository', [])
        
        download_score = min(4, len(downloads)) if downloads else 0
        link_score = min(3, len(links)) if links else 0
        repo_score = min(3, len(repos)) if repos else 0
        
        total_score = (download_score + link_score + repo_score) / 10 * max_score
        
        details = {
            'download_count': len(downloads),
            'link_count': len(links),
            'repository_count': len(repos)
        }
        
        return total_score, details
    
    def score_community(self, tool: Dict) -> Tuple[float, Dict]:
        """Score community information completeness."""
        max_score = self.weights['community']
        
        credits = tool.get('credit', [])
        contacts = tool.get('contact', [])
        
        credit_score = min(5, len(credits)) if credits else 0
        contact_score = min(5, len(contacts)) if contacts else 0
        
        total_score = (credit_score + contact_score) / 10 * max_score
        
        details = {
            'credit_count': len(credits),
            'contact_count': len(contacts)
        }
        
        return total_score, details
    
    def determine_tier(self, total_score: float) -> int:
        """Determine tier based on total score."""
        for tier, (min_score, max_score) in self.tier_thresholds.items():
            if min_score <= total_score <= max_score:
                return tier
        return 1  # Default to tier 1
    
    def score_tool(self, tool: Dict) -> Dict[str, Any]:
        """Calculate comprehensive score for a tool."""
        # Calculate component scores
        basic_score, basic_details = self.score_basic_info(tool)
        desc_score, desc_details = self.score_detailed_description(tool)
        tech_score, tech_details = self.score_technical_details(tool)
        doc_score, doc_details = self.score_documentation(tool)
        access_score, access_details = self.score_accessibility(tool)
        comm_score, comm_details = self.score_community(tool)
        
        # Calculate total score
        total_score = basic_score + desc_score + tech_score + doc_score + access_score + comm_score
        tier = self.determine_tier(total_score)
        
        return {
            'biotoolsID': tool.get('biotoolsID', 'unknown'),
            'name': tool.get('name', 'unknown'),
            'total_score': round(total_score, 2),
            'tier': tier,
            'scores': {
                'basic_info': round(basic_score, 2),
                'detailed_description': round(desc_score, 2),
                'technical_details': round(tech_score, 2),
                'documentation': round(doc_score, 2),
                'accessibility': round(access_score, 2),
                'community': round(comm_score, 2)
            },
            'details': {
                'basic_info': basic_details,
                'detailed_description': desc_details,
                'technical_details': tech_details,
                'documentation': doc_details,
                'accessibility': access_details,
                'community': comm_details
            }
        }
    
    def score_tools(self, tools: List[Dict]) -> List[Dict[str, Any]]:
        """Score multiple tools."""
        results = []
        for i, tool in enumerate(tools):
            try:
                result = self.score_tool(tool)
                result['index'] = i
                results.append(result)
            except Exception as e:
                logger.error(f"Failed to score tool {i}: {e}")
                results.append({
                    'index': i,
                    'biotoolsID': tool.get('biotoolsID', f'unknown_{i}'),
                    'error': str(e),
                    'total_score': 0,
                    'tier': 1
                })
        return results

# Initialize scorer and score the tools
scorer = QualityScorer()
print("🎯 Scoring tools for quality...")

scoring_results = scorer.score_tools(proteomics_tools)

print(f"✅ Scored {len(scoring_results)} tools")
print(f"📊 Average quality score: {np.mean([r.get('total_score', 0) for r in scoring_results]):.2f}/100")

## 5. Analyze Completeness Patterns Across Tool Collections

Now we'll aggregate and visualize completeness scores to identify trends, gaps, and patterns across different tool collections and domains.

In [ ]:
# Analyze completeness patterns
def analyze_completeness_patterns(results: List[Dict]) -> Dict[str, Any]:
    """Analyze patterns in field completeness."""
    # Extract scores and tiers
    scores = [r.get('total_score', 0) for r in results if 'error' not in r]
    tiers = [r.get('tier', 1) for r in results if 'error' not in r]
    
    # Count missing fields across all tools
    missing_fields = Counter()
    present_fields = Counter()
    
    for result in results:
        if 'error' not in result and 'details' in result:
            for section, details in result['details'].items():
                if isinstance(details, dict):
                    for field in details.get('missing_fields', []):
                        missing_fields[field] += 1
                    for field in details.get('present_fields', []):
                        present_fields[field] += 1
    
    return {
        'score_statistics': {
            'mean': np.mean(scores) if scores else 0,
            'median': np.median(scores) if scores else 0,
            'std': np.std(scores) if scores else 0,
            'min': np.min(scores) if scores else 0,
            'max': np.max(scores) if scores else 0
        },
        'tier_distribution': dict(Counter(tiers)),
        'missing_fields': dict(missing_fields),
        'present_fields': dict(present_fields),
        'total_tools': len([r for r in results if 'error' not in r])
    }

# Analyze patterns
patterns = analyze_completeness_patterns(scoring_results)

print("📈 COMPLETENESS ANALYSIS RESULTS")
print("=" * 40)
print(f"Total tools analyzed: {patterns['total_tools']}")
print(f"Average quality score: {patterns['score_statistics']['mean']:.2f}")
print(f"Score range: {patterns['score_statistics']['min']:.1f} - {patterns['score_statistics']['max']:.1f}")
print()

print("🏆 TIER DISTRIBUTION:")
for tier, count in sorted(patterns['tier_distribution'].items()):
    percentage = (count / patterns['total_tools']) * 100
    print(f"Tier {tier}: {count} tools ({percentage:.1f}%)")
print()

print("❌ MOST COMMONLY MISSING FIELDS:")
most_missing = sorted(patterns['missing_fields'].items(), key=lambda x: x[1], reverse=True)[:8]
for field, count in most_missing:
    percentage = (count / patterns['total_tools']) * 100
    print(f"- {field}: {count} tools ({percentage:.1f}%)")

# Create tier distribution visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Tier distribution pie chart
tier_counts = patterns['tier_distribution']
colors = [TIER_COLORS.get(tier, 'gray') for tier in sorted(tier_counts.keys())]
labels = [f'Tier {tier}' for tier in sorted(tier_counts.keys())]
values = [tier_counts[tier] for tier in sorted(tier_counts.keys())]

ax1.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title('Tool Quality Tier Distribution', fontsize=14, fontweight='bold')

# Score distribution histogram
scores = [r.get('total_score', 0) for r in scoring_results if 'error' not in r]
ax2.hist(scores, bins=15, color='skyblue', alpha=0.7, edgecolor='black')
ax2.axvline(np.mean(scores), color='red', linestyle='--', label=f'Mean: {np.mean(scores):.1f}')
ax2.set_xlabel('Quality Score')
ax2.set_ylabel('Number of Tools')
ax2.set_title('Distribution of Quality Scores', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Component score analysis
component_scores = defaultdict(list)
for result in scoring_results:
    if 'error' not in result and 'scores' in result:
        for component, score in result['scores'].items():
            component_scores[component].append(score)

# Create component comparison chart
fig, ax = plt.subplots(figsize=(12, 8))

components = list(component_scores.keys())
avg_scores = [np.mean(component_scores[comp]) for comp in components]
max_scores = [20, 15, 25, 20, 10, 10]  # Max possible scores for each component

x = np.arange(len(components))
width = 0.35

bars1 = ax.bar(x - width/2, avg_scores, width, label='Average Scores', color='lightblue', alpha=0.8)
bars2 = ax.bar(x + width/2, max_scores, width, label='Maximum Possible', color='lightcoral', alpha=0.8)

ax.set_xlabel('Quality Components')
ax.set_ylabel('Scores')
ax.set_title('Average Scores by Quality Component', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([comp.replace('_', ' ').title() for comp in components], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# Add value labels on bars
def add_value_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}', ha='center', va='bottom')

add_value_labels(bars1)
add_value_labels(bars2)

plt.tight_layout()
plt.show()

## 6. Integrate and Parse bio.tools Linter Output

We'll simulate the bio.tools linter functionality to identify structural and syntactic issues in tool metadata. In practice, you would integrate with the actual linter API or tool.

In [ ]:
class BioToolsLinter:
    """Simulated bio.tools linter for identifying structural issues."""
    
    def __init__(self):
        self.error_types = {
            'missing_required_field': {'severity': 'critical', 'weight': 10},
            'invalid_url': {'severity': 'major', 'weight': 5},
            'invalid_format': {'severity': 'major', 'weight': 5},
            'deprecated_field': {'severity': 'minor', 'weight': 2},
            'empty_field': {'severity': 'warning', 'weight': 1},
            'inconsistent_data': {'severity': 'major', 'weight': 5}
        }
    
    def _check_required_fields(self, tool: Dict) -> List[Dict]:
        """Check for missing required fields."""
        errors = []
        required_fields = ['biotoolsID', 'name', 'description']
        
        for field in required_fields:
            if not tool.get(field) or (isinstance(tool.get(field), str) and not tool.get(field).strip()):
                errors.append({
                    'type': 'missing_required_field',
                    'field': field,
                    'message': f"Required field '{field}' is missing or empty",
                    'severity': 'critical'
                })
        
        return errors
    
    def _check_url_validity(self, tool: Dict) -> List[Dict]:
        \"\"\"Check for invalid URLs (simplified check).\"\"\"
        errors = []
        url_fields = ['homepage']
        
        for field in url_fields:
            url = tool.get(field)
            if url and isinstance(url, str):
                if not (url.startswith('http://') or url.startswith('https://')):
                    errors.append({
                        'type': 'invalid_url',
                        'field': field,
                        'message': f"URL in '{field}' should start with http:// or https://",
                        'severity': 'major'
                    })
        
        return errors
    
    def _check_empty_arrays(self, tool: Dict) -> List[Dict]:
        \"\"\"Check for empty arrays that should contain data.\"\"\"
        errors = []
        array_fields = ['topic', 'function', 'toolType']
        
        for field in array_fields:
            value = tool.get(field)
            if isinstance(value, list) and len(value) == 0:
                errors.append({
                    'type': 'empty_field',
                    'field': field,
                    'message': f"Field '{field}' is an empty array",
                    'severity': 'warning'
                })
        
        return errors
    
    def _check_data_consistency(self, tool: Dict) -> List[Dict]:
        \"\"\"Check for data consistency issues.\"\"\"
        errors = []
        
        # Check if tool has functions but no operations
        functions = tool.get('function', [])
        if functions:
            for i, func in enumerate(functions):
                operations = func.get('operation', [])
                if not operations:
                    errors.append({
                        'type': 'inconsistent_data',
                        'field': f'function[{i}].operation',
                        'message': f"Function {i} has no operations defined",
                        'severity': 'major'
                    })
        
        return errors
    
    def lint_tool(self, tool: Dict) -> Dict[str, Any]:
        \"\"\"Run linter checks on a single tool.\"\"\"
        errors = []
        
        # Run all checks
        errors.extend(self._check_required_fields(tool))
        errors.extend(self._check_url_validity(tool))
        errors.extend(self._check_empty_arrays(tool))
        errors.extend(self._check_data_consistency(tool))
        
        # Calculate linter score
        total_penalty = sum(self.error_types[error['type']]['weight'] for error in errors)
        linter_score = max(0, 100 - total_penalty)
        
        # Categorize errors by severity
        error_counts = Counter(error['severity'] for error in errors)
        
        return {
            'biotoolsID': tool.get('biotoolsID', 'unknown'),
            'linter_score': linter_score,
            'total_errors': len(errors),
            'errors': errors,
            'error_counts': dict(error_counts),
            'has_critical_errors': error_counts.get('critical', 0) > 0
        }
    
    def lint_tools(self, tools: List[Dict]) -> List[Dict[str, Any]]:
        \"\"\"Run linter checks on multiple tools.\"\"\"
        results = []
        
        for i, tool in enumerate(tools):
            try:
                result = self.lint_tool(tool)
                result['index'] = i
                results.append(result)
            except Exception as e:
                logger.error(f"Failed to lint tool {i}: {e}")
                results.append({
                    'index': i,
                    'biotoolsID': tool.get('biotoolsID', f'unknown_{i}'),
                    'error': str(e),
                    'linter_score': 0,
                    'total_errors': 0
                })
        
        return results

# Initialize linter and run checks
linter = BioToolsLinter()
print(\"🔍 Running linter checks on tools...\")

linter_results = linter.lint_tools(proteomics_tools)

# Analyze linter results
total_tools = len(linter_results)
tools_with_errors = sum(1 for r in linter_results if r.get('total_errors', 0) > 0)
critical_errors = sum(1 for r in linter_results if r.get('has_critical_errors', False))
avg_linter_score = np.mean([r.get('linter_score', 0) for r in linter_results])

print(f\"✅ Linted {total_tools} tools\")
print(f\"⚠️  Tools with errors: {tools_with_errors} ({tools_with_errors/total_tools*100:.1f}%)\")
print(f\"🚨 Tools with critical errors: {critical_errors} ({critical_errors/total_tools*100:.1f}%)\")
print(f\"📊 Average linter score: {avg_linter_score:.2f}/100\")

# Show sample errors
print(\"\\n🚨 SAMPLE LINTER ERRORS:\")
for result in linter_results[:3]:
    if result.get('errors'):
        print(f\"\\n- {result['biotoolsID']}:\")
        for error in result['errors'][:2]:  # Show first 2 errors
            print(f\"  • {error['severity'].upper()}: {error['message']}\")

# Visualize linter results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Error distribution by severity
all_errors = []
for result in linter_results:
    all_errors.extend(result.get('errors', []))

if all_errors:
    severity_counts = Counter(error['severity'] for error in all_errors)
    ax1.bar(severity_counts.keys(), severity_counts.values(), 
            color=['red', 'orange', 'yellow', 'blue'])
    ax1.set_title('Distribution of Linter Errors by Severity', fontweight='bold')
    ax1.set_ylabel('Number of Errors')
    
    # Add value labels
    for i, (severity, count) in enumerate(severity_counts.items()):
        ax1.text(i, count + 0.1, str(count), ha='center', va='bottom')

# Linter score distribution
linter_scores = [r.get('linter_score', 0) for r in linter_results]
ax2.hist(linter_scores, bins=10, color='lightgreen', alpha=0.7, edgecolor='black')
ax2.axvline(np.mean(linter_scores), color='red', linestyle='--', 
           label=f'Mean: {np.mean(linter_scores):.1f}')
ax2.set_xlabel('Linter Score')
ax2.set_ylabel('Number of Tools')
ax2.set_title('Distribution of Linter Scores', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()"

## 7. Merge Completeness Scores with Linter Diagnostics

Now we'll combine the completeness scoring results with linter diagnostics to create an integrated quality assessment for each tool.

In [ ]:
def merge_quality_assessments(scoring_results: List[Dict], linter_results: List[Dict]) -> List[Dict]:
    \"\"\"Merge completeness scores with linter diagnostics.\"\"\"
    merged_results = []
    
    # Create lookup for linter results
    linter_lookup = {result['biotoolsID']: result for result in linter_results}
    
    for score_result in scoring_results:
        biotoolsid = score_result.get('biotoolsID', 'unknown')
        linter_result = linter_lookup.get(biotoolsid, {})
        
        # Calculate composite quality score
        completeness_score = score_result.get('total_score', 0)
        linter_score = linter_result.get('linter_score', 100)  # Default to 100 if no linter issues
        
        # Weighted composite score (70% completeness, 30% linter)
        composite_score = (completeness_score * 0.7) + (linter_score * 0.3)
        
        # Determine final quality tier based on composite score
        final_tier = 1
        if composite_score >= 80:
            final_tier = 5
        elif composite_score >= 60:
            final_tier = 4
        elif composite_score >= 40:
            final_tier = 3
        elif composite_score >= 20:
            final_tier = 2
        
        # Create merged result
        merged_result = {
            'biotoolsID': biotoolsid,
            'name': score_result.get('name', 'unknown'),
            'completeness_score': completeness_score,
            'linter_score': linter_score,
            'composite_score': round(composite_score, 2),
            'completeness_tier': score_result.get('tier', 1),
            'final_tier': final_tier,
            'total_errors': linter_result.get('total_errors', 0),
            'has_critical_errors': linter_result.get('has_critical_errors', False),
            'error_counts': linter_result.get('error_counts', {}),
            'component_scores': score_result.get('scores', {}),
            'improvement_priority': 'high' if composite_score < 40 else 'medium' if composite_score < 70 else 'low'
        }
        
        merged_results.append(merged_result)
    
    return merged_results

# Merge the results
print(\"🔗 Merging completeness scores with linter diagnostics...\")
integrated_results = merge_quality_assessments(scoring_results, linter_results)

# Analyze integrated results
total_tools = len(integrated_results)
high_priority = sum(1 for r in integrated_results if r['improvement_priority'] == 'high')
medium_priority = sum(1 for r in integrated_results if r['improvement_priority'] == 'medium')
low_priority = sum(1 for r in integrated_results if r['improvement_priority'] == 'low')

avg_composite_score = np.mean([r['composite_score'] for r in integrated_results])
avg_completeness = np.mean([r['completeness_score'] for r in integrated_results])
avg_linter = np.mean([r['linter_score'] for r in integrated_results])

print(f\"✅ Integrated assessment completed for {total_tools} tools\")
print(f\"📊 Average composite score: {avg_composite_score:.2f}/100\")
print(f\"📈 Average completeness score: {avg_completeness:.2f}/100\")
print(f\"🔍 Average linter score: {avg_linter:.2f}/100\")
print()
print(\"🎯 IMPROVEMENT PRIORITIES:\")
print(f\"🔴 High priority (score < 40): {high_priority} tools ({high_priority/total_tools*100:.1f}%)\")
print(f\"🟡 Medium priority (40-70): {medium_priority} tools ({medium_priority/total_tools*100:.1f}%)\")
print(f\"🟢 Low priority (> 70): {low_priority} tools ({low_priority/total_tools*100:.1f}%)\")

# Create comprehensive visualization
fig = plt.figure(figsize=(16, 12))

# Create a 2x3 subplot layout
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1])

# 1. Completeness vs Linter scores scatter plot
ax1 = fig.add_subplot(gs[0, 0])
completeness_scores = [r['completeness_score'] for r in integrated_results]
linter_scores = [r['linter_score'] for r in integrated_results]
colors = [TIER_COLORS[r['final_tier']] for r in integrated_results]

scatter = ax1.scatter(completeness_scores, linter_scores, c=colors, alpha=0.7, s=50)
ax1.set_xlabel('Completeness Score')
ax1.set_ylabel('Linter Score')
ax1.set_title('Completeness vs Linter Scores', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Add diagonal line for reference
ax1.plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Perfect correlation')
ax1.legend()

# 2. Final tier distribution
ax2 = fig.add_subplot(gs[0, 1])
final_tiers = [r['final_tier'] for r in integrated_results]
tier_counts = Counter(final_tiers)
colors = [TIER_COLORS[tier] for tier in sorted(tier_counts.keys())]
labels = [f'Tier {tier}' for tier in sorted(tier_counts.keys())]
values = [tier_counts[tier] for tier in sorted(tier_counts.keys())]

ax2.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Final Quality Tier Distribution', fontweight='bold')

# 3. Composite score distribution
ax3 = fig.add_subplot(gs[1, :])
composite_scores = [r['composite_score'] for r in integrated_results]
ax3.hist(composite_scores, bins=15, color='lightblue', alpha=0.7, edgecolor='black')
ax3.axvline(np.mean(composite_scores), color='red', linestyle='--', 
           label=f'Mean: {np.mean(composite_scores):.1f}')
ax3.axvline(40, color='orange', linestyle=':', label='High Priority Threshold')
ax3.axvline(70, color='green', linestyle=':', label='Low Priority Threshold')
ax3.set_xlabel('Composite Quality Score')
ax3.set_ylabel('Number of Tools')
ax3.set_title('Distribution of Composite Quality Scores', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Improvement priority breakdown
ax4 = fig.add_subplot(gs[2, 0])
priorities = ['High', 'Medium', 'Low']
priority_counts = [high_priority, medium_priority, low_priority]
priority_colors = ['red', 'orange', 'green']

bars = ax4.bar(priorities, priority_counts, color=priority_colors, alpha=0.7)
ax4.set_ylabel('Number of Tools')
ax4.set_title('Tools by Improvement Priority', fontweight='bold')

# Add value labels on bars
for bar, count in zip(bars, priority_counts):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'{count}\\n({count/total_tools*100:.1f}%)', 
             ha='center', va='bottom')

# 5. Score comparison by component
ax5 = fig.add_subplot(gs[2, 1])
component_names = ['Completeness', 'Linter', 'Composite']
component_scores = [avg_completeness, avg_linter, avg_composite_score]
component_colors = ['blue', 'green', 'purple']

bars = ax5.bar(component_names, component_scores, color=component_colors, alpha=0.7)
ax5.set_ylabel('Average Score')
ax5.set_title('Average Scores by Assessment Type', fontweight='bold')
ax5.set_ylim(0, 100)

# Add value labels
for bar, score in zip(bars, component_scores):
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{score:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Create detailed results table
results_df = pd.DataFrame(integrated_results)
print(\"\\n📋 DETAILED RESULTS TABLE (Top 10 tools by composite score):\")
display_cols = ['name', 'completeness_score', 'linter_score', 'composite_score', 
                'final_tier', 'total_errors', 'improvement_priority']
top_tools = results_df.nlargest(10, 'composite_score')[display_cols]
print(top_tools.to_string(index=False))"

## 8. Generate Visual Summaries (Radar Charts, Heatmaps)

We'll create comprehensive visual summaries including radar charts and heatmaps to effectively communicate completeness metrics and linter findings across tools and domains.

In [ ]:
# Create advanced visualizations using Plotly for interactive charts

# 1. Radar Chart - Average Component Scores by Tier
def create_radar_chart_by_tier(integrated_results):
    \"\"\"Create radar chart showing average component scores by tier.\"\"\"
    
    # Group results by tier
    tier_data = defaultdict(lambda: defaultdict(list))
    
    for result in integrated_results:
        tier = result['final_tier']
        for component, score in result['component_scores'].items():
            tier_data[tier][component].append(score)
    
    # Calculate averages
    tier_averages = {}
    for tier, components in tier_data.items():
        tier_averages[tier] = {comp: np.mean(scores) for comp, scores in components.items()}
    
    # Create radar chart
    fig = go.Figure()
    
    components = ['basic_info', 'detailed_description', 'technical_details', 
                 'documentation', 'accessibility', 'community']
    component_labels = [comp.replace('_', ' ').title() for comp in components]
    
    for tier in sorted(tier_averages.keys()):
        values = [tier_averages[tier].get(comp, 0) for comp in components]
        values.append(values[0])  # Close the radar chart
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=component_labels + [component_labels[0]],
            fill='toself',
            name=f'Tier {tier}',
            line_color=TIER_COLORS[tier]
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 25]  # Max possible score for any component
            )
        ),
        showlegend=True,
        title=\"Average Component Scores by Quality Tier\",
        width=600,
        height=500
    )
    
    return fig

# 2. Heatmap - Field Completeness Matrix
def create_completeness_heatmap(integrated_results):
    \"\"\"Create heatmap showing field completeness patterns.\"\"\"
    
    # Extract field presence information
    field_presence = defaultdict(lambda: defaultdict(int))
    
    for result in scoring_results:  # Use original scoring results for detailed field info
        tier = result.get('tier', 1)
        details = result.get('details', {})
        
        for section, section_details in details.items():
            if isinstance(section_details, dict):
                present_fields = section_details.get('present_fields', [])
                missing_fields = section_details.get('missing_fields', [])
                
                for field in present_fields:
                    field_presence[field][f'Tier {tier}'] += 1
                for field in missing_fields:
                    # Count as 0 for missing fields
                    pass
    
    # Convert to DataFrame for heatmap
    if field_presence:
        # Get all fields and tiers
        all_fields = list(field_presence.keys())[:15]  # Limit for readability
        all_tiers = [f'Tier {i}' for i in range(1, 6)]
        
        # Create matrix
        matrix_data = []
        for field in all_fields:
            row = []
            for tier in all_tiers:
                count = field_presence[field][tier]
                total_tools_in_tier = sum(1 for r in integrated_results if r['final_tier'] == int(tier.split()[1]))
                percentage = (count / total_tools_in_tier * 100) if total_tools_in_tier > 0 else 0
                row.append(percentage)
            matrix_data.append(row)
        
        # Create heatmap
        fig = go.Figure(data=go.Heatmap(
            z=matrix_data,
            x=all_tiers,
            y=all_fields,
            colorscale='RdYlGn',
            text=[[f'{val:.1f}%' for val in row] for row in matrix_data],
            texttemplate='%{text}',
            textfont={\"size\": 10},
            hoverongaps=False
        ))
        
        fig.update_layout(
            title='Field Completeness by Quality Tier (%)',
            xaxis_title='Quality Tier',
            yaxis_title='Metadata Fields',
            width=800,
            height=600
        )
        
        return fig
    
    return None

# 3. Interactive Scatter Plot - Tool Quality Landscape
def create_quality_landscape(integrated_results):
    \"\"\"Create interactive scatter plot of tool quality landscape.\"\"\"
    
    df = pd.DataFrame(integrated_results)
    
    fig = px.scatter(
        df, 
        x='completeness_score', 
        y='linter_score',
        size='composite_score',
        color='final_tier',
        hover_data=['name', 'total_errors', 'improvement_priority'],
        color_discrete_map={tier: color for tier, color in TIER_COLORS.items()},
        title='Tool Quality Landscape: Completeness vs Linter Scores'
    )
    
    # Add diagonal reference line
    fig.add_shape(
        type=\"line\",
        x0=0, y0=0, x1=100, y1=100,
        line=dict(color=\"gray\", width=2, dash=\"dash\"),
    )
    
    fig.update_layout(
        xaxis_title='Completeness Score',
        yaxis_title='Linter Score',
        width=800,
        height=600
    )
    
    return fig

# Generate all visualizations
print(\"🎨 Creating advanced visualizations...\")

# Radar chart
radar_fig = create_radar_chart_by_tier(integrated_results)
radar_fig.show()

# Heatmap
heatmap_fig = create_completeness_heatmap(integrated_results)
if heatmap_fig:
    heatmap_fig.show()
else:
    print(\"⚠️ Heatmap could not be generated due to insufficient data\")

# Quality landscape scatter plot
landscape_fig = create_quality_landscape(integrated_results)
landscape_fig.show()

# Summary statistics visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. Error types distribution
all_errors = []
for result in linter_results:
    all_errors.extend(result.get('errors', []))

if all_errors:
    error_types = Counter(error['type'] for error in all_errors)
    ax1.barh(list(error_types.keys()), list(error_types.values()), color='coral')
    ax1.set_title('Distribution of Error Types', fontweight='bold')
    ax1.set_xlabel('Number of Occurrences')

# 2. Quality improvement potential
improvement_potential = []
for result in integrated_results:
    max_possible = 100
    current_score = result['composite_score']
    potential = max_possible - current_score
    improvement_potential.append(potential)

ax2.hist(improvement_potential, bins=15, color='lightgreen', alpha=0.7, edgecolor='black')
ax2.set_title('Quality Improvement Potential Distribution', fontweight='bold')
ax2.set_xlabel('Improvement Potential (points)')
ax2.set_ylabel('Number of Tools')
ax2.grid(True, alpha=0.3)

# 3. Component score correlations
component_scores = pd.DataFrame([r['component_scores'] for r in integrated_results])
correlation_matrix = component_scores.corr()

im = ax3.imshow(correlation_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax3.set_xticks(range(len(correlation_matrix.columns)))
ax3.set_yticks(range(len(correlation_matrix.columns)))
ax3.set_xticklabels([col.replace('_', '\\n') for col in correlation_matrix.columns], rotation=45)
ax3.set_yticklabels([col.replace('_', '\\n') for col in correlation_matrix.columns])
ax3.set_title('Component Score Correlations', fontweight='bold')

# Add correlation values
for i in range(len(correlation_matrix.columns)):
    for j in range(len(correlation_matrix.columns)):
        ax3.text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}', 
                ha='center', va='center', color='white' if abs(correlation_matrix.iloc[i, j]) > 0.5 else 'black')

plt.colorbar(im, ax=ax3, shrink=0.8)

# 4. Quality tier transition analysis
tier_changes = []
for result in integrated_results:
    completeness_tier = None
    # Find corresponding completeness tier from scoring results
    for score_result in scoring_results:
        if score_result['biotoolsID'] == result['biotoolsID']:
            completeness_tier = score_result['tier']
            break
    
    if completeness_tier:
        tier_change = result['final_tier'] - completeness_tier
        tier_changes.append(tier_change)

if tier_changes:
    change_counts = Counter(tier_changes)
    changes = list(change_counts.keys())
    counts = list(change_counts.values())
    
    colors = ['red' if c < 0 else 'green' if c > 0 else 'gray' for c in changes]
    ax4.bar(changes, counts, color=colors, alpha=0.7)
    ax4.set_title('Tier Changes After Linter Integration', fontweight='bold')
    ax4.set_xlabel('Tier Change (Final - Completeness)')
    ax4.set_ylabel('Number of Tools')
    ax4.grid(True, alpha=0.3)
    
    # Add labels
    for change, count in zip(changes, counts):
        ax4.text(change, count + 0.1, str(count), ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(\"✅ Advanced visualizations completed!\")"

## 9. Summarize and Report Frequently Missing or Malformed Attributes

We'll systematically analyze and report the most commonly missing or malformed metadata fields, providing actionable insights for improving tool annotations.

In [ ]:
def analyze_missing_attributes(scoring_results, linter_results, integrated_results):
    \"\"\"Comprehensive analysis of missing and malformed attributes.\"\"\"
    
    analysis = {
        'missing_fields_summary': {},
        'error_patterns': {},
        'critical_gaps': {},
        'tier_specific_issues': {},
        'improvement_recommendations': []
    }
    
    # 1. Analyze missing fields from scoring results
    all_missing_fields = Counter()
    tier_missing_fields = defaultdict(lambda: Counter())
    
    for result in scoring_results:
        if 'error' not in result:
            tier = result.get('tier', 1)
            details = result.get('details', {})
            
            for section, section_details in details.items():
                if isinstance(section_details, dict):
                    missing_fields = section_details.get('missing_fields', [])
                    for field in missing_fields:
                        all_missing_fields[field] += 1
                        tier_missing_fields[tier][field] += 1
    
    analysis['missing_fields_summary'] = dict(all_missing_fields.most_common(15))
    
    # 2. Analyze error patterns from linter results
    error_patterns = Counter()
    error_by_severity = defaultdict(Counter)
    
    for result in linter_results:
        errors = result.get('errors', [])
        for error in errors:
            error_type = error.get('type', 'unknown')
            severity = error.get('severity', 'unknown')
            error_patterns[error_type] += 1
            error_by_severity[severity][error_type] += 1
    
    analysis['error_patterns'] = dict(error_patterns.most_common(10))
    
    # 3. Identify critical gaps (fields missing in >70% of tools)
    total_tools = len([r for r in scoring_results if 'error' not in r])
    critical_threshold = total_tools * 0.7
    
    critical_gaps = {field: count for field, count in all_missing_fields.items() 
                    if count >= critical_threshold}
    analysis['critical_gaps'] = critical_gaps
    
    # 4. Tier-specific issues
    for tier in range(1, 6):
        tier_issues = dict(tier_missing_fields[tier].most_common(5))
        analysis['tier_specific_issues'][f'tier_{tier}'] = tier_issues
    
    # 5. Generate improvement recommendations
    recommendations = []
    
    # High-priority recommendations based on critical gaps
    for field, count in critical_gaps.items():
        percentage = (count / total_tools) * 100
        recommendations.append(f\"CRITICAL: '{field}' is missing in {count} tools ({percentage:.1f}%). This field should be prioritized for completion.\")
    
    # Tier-specific recommendations
    if tier_missing_fields[1]:
        most_missing_tier1 = tier_missing_fields[1].most_common(3)
        recommendations.append(f\"Tier 1 tools most commonly lack: {', '.join([field for field, _ in most_missing_tier1])}. Focus on basic metadata completion.\")
    
    if tier_missing_fields[5]:
        most_missing_tier5 = tier_missing_fields[5].most_common(3)
        recommendations.append(f\"Even Tier 5 tools often lack: {', '.join([field for field, _ in most_missing_tier5])}. These represent opportunities for excellence.\")
    
    # Error-based recommendations
    if 'missing_required_field' in error_patterns:
        recommendations.append(f\"Schema validation shows {error_patterns['missing_required_field']} instances of missing required fields. Implement validation checks.\")
    
    if 'invalid_url' in error_patterns:
        recommendations.append(f\"Found {error_patterns['invalid_url']} invalid URLs. Implement URL validation and correction.\")
    
    analysis['improvement_recommendations'] = recommendations
    
    return analysis

# Perform comprehensive analysis
print(\"📊 Analyzing missing and malformed attributes...\")
attribute_analysis = analyze_missing_attributes(scoring_results, linter_results, integrated_results)

# Display results
print(\"\\n🚨 CRITICAL FINDINGS\")
print(\"=\" * 50)

print(\"\\n📈 MOST COMMONLY MISSING FIELDS:\")
for field, count in list(attribute_analysis['missing_fields_summary'].items())[:10]:
    total_tools = len([r for r in scoring_results if 'error' not in r])
    percentage = (count / total_tools) * 100
    print(f\"• {field}: {count} tools ({percentage:.1f}%)\")

print(\"\\n🔍 MOST COMMON ERROR PATTERNS:\")
for error_type, count in list(attribute_analysis['error_patterns'].items())[:8]:
    print(f\"• {error_type.replace('_', ' ').title()}: {count} occurrences\")

print(\"\\n🚨 CRITICAL GAPS (>70% of tools missing):\")
if attribute_analysis['critical_gaps']:
    for field, count in attribute_analysis['critical_gaps'].items():
        total_tools = len([r for r in scoring_results if 'error' not in r])
        percentage = (count / total_tools) * 100
        print(f\"• {field}: {count} tools ({percentage:.1f}%)\")
else:
    print(\"• No fields are missing in >70% of tools (Good news!)\")

print(\"\\n💡 KEY IMPROVEMENT RECOMMENDATIONS:\")
for i, recommendation in enumerate(attribute_analysis['improvement_recommendations'][:5], 1):
    print(f\"{i}. {recommendation}\")

# Create comprehensive visualization of findings
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(18, 12))

# 1. Top missing fields
missing_fields = list(attribute_analysis['missing_fields_summary'].items())[:12]
if missing_fields:
    fields, counts = zip(*missing_fields)
    total_tools = len([r for r in scoring_results if 'error' not in r])
    percentages = [(count / total_tools) * 100 for count in counts]
    
    bars = ax1.barh(range(len(fields)), percentages, color='coral', alpha=0.7)
    ax1.set_yticks(range(len(fields)))
    ax1.set_yticklabels([f.replace('_', ' ').title() for f in fields])
    ax1.set_xlabel('Percentage of Tools Missing Field')
    ax1.set_title('Most Commonly Missing Fields', fontweight='bold', fontsize=14)
    ax1.grid(True, alpha=0.3)
    
    # Add percentage labels
    for i, (bar, pct) in enumerate(zip(bars, percentages)):
        ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                f'{pct:.1f}%', va='center', ha='left')

# 2. Error severity distribution
all_errors = []
for result in linter_results:
    all_errors.extend(result.get('errors', []))

if all_errors:
    severity_counts = Counter(error['severity'] for error in all_errors)
    severities = list(severity_counts.keys())
    counts = list(severity_counts.values())
    colors = {'critical': 'red', 'major': 'orange', 'minor': 'yellow', 'warning': 'lightblue'}
    bar_colors = [colors.get(sev, 'gray') for sev in severities]
    
    bars = ax2.bar(severities, counts, color=bar_colors, alpha=0.8)
    ax2.set_ylabel('Number of Errors')
    ax2.set_title('Error Distribution by Severity', fontweight='bold', fontsize=14)
    ax2.grid(True, alpha=0.3)
    
    # Add count labels
    for bar, count in zip(bars, counts):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(count), ha='center', va='bottom')

# 3. Missing fields by tier
tier_missing_data = []
tiers = range(1, 6)
top_missing_fields = list(attribute_analysis['missing_fields_summary'].keys())[:8]

for field in top_missing_fields:
    field_data = []
    for tier in tiers:
        tier_issues = attribute_analysis['tier_specific_issues'].get(f'tier_{tier}', {})
        count = tier_issues.get(field, 0)
        # Calculate percentage of tools in this tier missing this field
        tier_tools = sum(1 for r in integrated_results if r['final_tier'] == tier)
        percentage = (count / tier_tools * 100) if tier_tools > 0 else 0
        field_data.append(percentage)
    tier_missing_data.append(field_data)

if tier_missing_data:
    im = ax3.imshow(tier_missing_data, cmap='Reds', aspect='auto')
    ax3.set_xticks(range(len(tiers)))
    ax3.set_yticks(range(len(top_missing_fields)))
    ax3.set_xticklabels([f'Tier {t}' for t in tiers])
    ax3.set_yticklabels([f.replace('_', ' ').title() for f in top_missing_fields])
    ax3.set_title('Missing Fields by Quality Tier (%)', fontweight='bold', fontsize=14)
    
    # Add percentage text
    for i in range(len(top_missing_fields)):
        for j in range(len(tiers)):
            text = ax3.text(j, i, f'{tier_missing_data[i][j]:.0f}%',
                           ha='center', va='center', color='white' if tier_missing_data[i][j] > 50 else 'black')
    
    plt.colorbar(im, ax=ax3, shrink=0.8)

# 4. Quality improvement impact analysis
improvement_impact = []
field_importance = {
    'homepage': 8, 'license': 9, 'documentation': 10, 'publication': 10,
    'contact': 7, 'toolType': 8, 'language': 6, 'operatingSystem': 5,
    'version': 7, 'download': 6, 'repository': 5, 'credit': 4
}

for field, missing_count in list(attribute_analysis['missing_fields_summary'].items())[:10]:
    importance = field_importance.get(field, 5)  # Default importance
    total_tools = len([r for r in scoring_results if 'error' not in r])
    potential_impact = (missing_count / total_tools) * importance * 100
    improvement_impact.append((field, potential_impact))

improvement_impact.sort(key=lambda x: x[1], reverse=True)

if improvement_impact:
    fields, impacts = zip(*improvement_impact)
    bars = ax4.barh(range(len(fields)), impacts, color='green', alpha=0.7)
    ax4.set_yticks(range(len(fields)))
    ax4.set_yticklabels([f.replace('_', ' ').title() for f in fields])
    ax4.set_xlabel('Improvement Impact Score')
    ax4.set_title('Potential Impact of Field Completion', fontweight='bold', fontsize=14)
    ax4.grid(True, alpha=0.3)
    
    # Add impact scores
    for i, (bar, impact) in enumerate(zip(bars, impacts)):
        ax4.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                f'{impact:.0f}', va='center', ha='left')

plt.tight_layout()
plt.show()

# Generate summary report
print(\"\\n\" + \"=\" * 80)
print(\"COMPREHENSIVE ATTRIBUTE ANALYSIS SUMMARY\")
print(\"=\" * 80)
print(f\"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\")
print(f\"Total Tools Analyzed: {len([r for r in scoring_results if 'error' not in r])}\")
print(f\"Total Errors Found: {sum(len(r.get('errors', [])) for r in linter_results)}\")
print(f\"Tools with Critical Errors: {sum(1 for r in linter_results if r.get('has_critical_errors', False))}\")
print()

print(\"KEY STATISTICS:\")
total_missing = sum(attribute_analysis['missing_fields_summary'].values())
print(f\"• Total missing field instances: {total_missing}\")
print(f\"• Average missing fields per tool: {total_missing / len([r for r in scoring_results if 'error' not in r]):.1f}\")
print(f\"• Fields with >50% missing rate: {sum(1 for count in attribute_analysis['missing_fields_summary'].values() if count > len([r for r in scoring_results if 'error' not in r]) * 0.5)}\")

print(\"\\n✅ Analysis completed successfully!\")"

## 10. Draft Proposed Revisions to Tool Information Standards

Based on our empirical findings, we'll draft evidence-based recommendations for modifying and clarifying the Tool Information Standards to improve the overall quality of bio.tools annotations.